# Initialize Truveta SDK

In [1]:
from truveta.study import Client, OutputMode, display_df
import pyspark.pandas as ps
import pandas as pd
import numpy as np

In [2]:
# Use only one statement below and comment out whichever you are not using.
client = Client(output_mode = OutputMode.PandasOnSpark)
#client = Client(output_mode = OutputMode.PySpark)

In [3]:
study = client.get_study()
# Use only one statement below and comment out whichever you are not using.
population = study.get_population(title = "Control Group")
#population = study.get_population(id = "ps-j6txx6453youzlly45sb5uc3n4")
# population
# Get latest completed active snapshot.
snapshot = population.get_latest_snapshot()
#snapshot
# Show tables in the snapshot.
#snapshot.get_tables()

In [4]:
## Running the decode_concepts function - use for match the concept code with the actual name
from typing import overload
import pyspark.pandas as ps
import pandas as pd
from pyspark.sql import DataFrame

@overload
def decode_concepts(df: pd.DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> pd.DataFrame: ...
@overload
def decode_concepts(df: ps.DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> ps.DataFrame: ...
@overload
def decode_concepts(df: DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> DataFrame: ...
def decode_concepts(df: pd.DataFrame | ps.DataFrame | DataFrame, drop_concepts: bool = True, columns: list[str] | None = None):
    """
    decodes the top level *ConceptId columns within the given data frame and derives new names (without the ConceptId suffix). 
    It assume that every column ending with `ConceptId` which is an integer or float is a concept column
    :param df the data frame to decode
    :param drop_concepts whether to drop *ConceptId columns (default true)
    :param columns optional direct list of columns to decode disabling the auto infering
    :returns the enhanced data frame
    """
    def should_decode(col: str, dtype: str) -> bool:
        if columns:
            return col in columns
        return col.endswith('ConceptId') and str(dtype) in ('int', 'int32', 'float64', 'float32')
    
    column_names = set(df.columns)

    def target_col(col: str) -> str:
        name = col.removesuffix('ConceptId')
        if name == col and drop_concepts:
            # return the same name
            return name
        while name in column_names:
            name = f"{name}Name"
        return name
    
    def safe_name(name: str) -> str:
        while name in column_names:
            name = f"{name}_tmp"
        return name

    final_order: list[str] = []
    
    if isinstance(df, pd.DataFrame):
        lookup = None
        for col, dtype in zip(df.columns, df.dtypes):
            if should_decode(col, str(dtype)):
                name = target_col(col)
                column_names.add(name)
                if lookup is None:
                    # lazy lookup
                    lookup = spark.sql("SELECT ConceptId, ConceptName FROM Concept").toPandas().set_index("ConceptId").ConceptName
                df[name] = df[col].map(lookup)
                if drop_concepts and name != col:
                    df = df.drop(columns=[col])
                else:
                    final_order.append(col) # keep original
                final_order.append(name)
            else:
                final_order.append(col)
        return df[final_order]

    return_pandas = False
    if isinstance(df, ps.DataFrame):
        return_pandas = True
        df = df.to_spark()
    
    concepts_s = spark.sql("SELECT ConceptId, ConceptName FROM Concept").cache()
    for col, dtype in df.dtypes:
        if should_decode(col, str(dtype)):
            name = target_col(col)
            column_names.add(name)
            if col == name and drop_concepts:
                # need to write to the same name, rename old one
                tmp_name = safe_name(col)
                df = df.withColumnRenamed(col, tmp_name).join(concepts_s.withColumnRenamed("ConceptId", tmp_name).withColumnRenamed("ConceptName", name), on=tmp_name, how="left").drop(tmp_name)
            else:
                df = df.join(concepts_s.withColumnRenamed("ConceptId", col).withColumnRenamed("ConceptName", name), on=col, how="left")
                if drop_concepts:
                    df = df.drop(col)
                else:
                    final_order.append(col)
            final_order.append(name)
        else:
            final_order.append(col)
    df = df.select(final_order)
    return df.pandas_api() if return_pandas else df

In [5]:
def match_code(df, codes_df):
    code_name = decode_concepts(df)
    case_names = codes_df.ConceptName.to_pandas().tolist()
    code_name = code_name[code_name['Code'].isin(case_names)]
    return code_name

### Removing the Excluded conditions

In [6]:
# getting procedureNotCodes
ectopics_code = snapshot.codeset_from_prose(url = "/definitions/delivery", variable_name = "ectopic_code")
ectopics_pro = snapshot.load_filtered_table("Procedure", ectopics_code, view_name = 'tbl_ectopics_pro')
print(ectopics_pro['PersonId'].nunique())
ectopics_con = snapshot.load_filtered_table("Condition", ectopics_code, view_name = 'tbl_ectopics_con')
print(ectopics_con['PersonId'].nunique())

In [7]:
df = ps.sql("SELECT p.PersonId, p.RecordedDateTime, p.StartDateTime, pm.* FROM tbl_ectopics_pro p JOIN ProcedureCodeConceptMap pm on p.CodeConceptMapId = pm.Id").to_pandas()
ectopics_pro = match_code(df, ectopics_code)

df = ps.sql("SELECT p.PersonId, p.RecordedDateTime, pm.* FROM tbl_ectopics_con p JOIN ConditionCodeConceptMap pm on p.CodeConceptMapId = pm.Id").to_pandas()
ectopics_con = match_code(df, ectopics_code)
ectopics = pd.concat([ectopics_pro, ectopics_con])
ectopics.PersonId.nunique()

In [8]:
multiple_code = snapshot.codeset_from_prose(url = "/definitions/delivery", variable_name = "multiple_code")
multiple_df = snapshot.load_filtered_table("Condition", multiple_code, view_name = 'tbl_multiple')
print(multiple_df['PersonId'].nunique())
df = ps.sql("SELECT p.PersonId, p.RecordedDateTime, pm.* FROM tbl_multiple p JOIN ConditionCodeConceptMap pm on p.CodeConceptMapId = pm.Id").to_pandas()
multiple_df = match_code(df, multiple_code)
#multiple_df.PersonId.nunique()

# Getting the Tables ready

### Getting Weight - Method From the Enablement Study

In [9]:
sql = """
SELECT 
* 
FROM Concept
"""

select_columns = ["ConceptId", "ConceptName"]
#* select columns needed to improve performance
concept = snapshot.load_sql_table(sql, view_name = 'tbl_concept')[["ConceptId", "ConceptName"]]
#display_df(concept)

In [10]:
weight_codes_s  = snapshot.codeset("LOINC", 'selfAndDescendants',
"58229-6",
  "18833-4",
  "29463-7",
  "3141-9",
  "3142-7",
  "8341-0",
  "8349-3",
  "8350-1",
  "8351-9")

study.create_view(weight_codes_s, 
   view_name = 'tbl_weight_codes_s')

In [11]:
lab_weights = snapshot.load_filtered_table("LabResult",
   weight_codes_s, view_name = 'tbl_lab_weights')
   # select and rename columns
lab_weights = lab_weights[['PersonId',
                               'EffectiveDateTime', 
                               'RecordedDateTime', 
                               'NormalizedValueConceptId', 
                               'NormalizedValueNumeric', 
                               'NormalizedValueUOMConceptId', 
                               'StatusConceptId', 
                               'EncounterId']]
display_df(lab_weights)

obs_weights = snapshot.load_filtered_table("Observation",
   weight_codes_s, view_name = 'tbl_obs_weights')

# select and rename columns

# select and rename columns
obs_weights = obs_weights[['PersonId',
                               'EffectiveDateTime', 
                               'RecordedDateTime', 
                               'NormalizedValueConceptId', 
                               'NormalizedValueNumeric', 
                               'NormalizedValueUOMConceptId', 
                               'StatusConceptId', 
                               'EncounterId']]

all_weights = ps.concat([obs_weights, lab_weights], 
   ignore_index=True)


In [12]:
all_weights = all_weights.merge(concept, 
    how='left', 
    left_on='NormalizedValueConceptId', 
    right_on='ConceptId') \
    .rename(columns={'ConceptName': 'NormalizedValueConcept'}) \
    .drop(columns=['ConceptId'])

all_weights = all_weights.merge(concept, 
   how='left', 
   left_on='NormalizedValueUOMConceptId', 
   right_on='ConceptId') \
   .rename(columns={'ConceptName': 'NormalizedValueUOMConcept'}) \
   .drop(columns=['ConceptId'])

all_weights = all_weights.merge(concept, 
   how='left', 
   left_on='StatusConceptId', 
   right_on='ConceptId') \
   .rename(columns={'ConceptName': 'StatusConcept'}) \
   .drop(columns=['ConceptId'])

In [13]:
units_table = all_weights.groupby('NormalizedValueUOMConcept')\
   .size().reset_index()

# take these out
# Define a list of units to filter out
excluded_units = [
    'per liter', 'per meter', 'billion per liter',
    'centimeter', 'degree', 'foot (US)', 'heart beats per minute', 
    'inches', 'liter', 'liter per minute', 'lumen', 
    'meter', 'millimeter mercury column', 'millivolt', 
    'minute', 'per hour', 'per meter', 'per liter', 
    'percent', 'second', 'week', 'Each', 'Inches', 
    'inch (international)', 'each'
]

# Filter the DataFrame
all_weights = all_weights[~all_weights['NormalizedValueUOMConcept'].isin(excluded_units)]

### Make unknowns the same variable: 

unknown_mapping = {
    'No Information': 'unknown',
    'Field has not been mapped': 'unknown',
    'Field is not present in source': 'unknown',
    'Invalid': 'unknown'
}

# Use replace to handle the mapping
all_weights['NormalizedValueUOMConcept'] = \
   all_weights['NormalizedValueUOMConcept'].replace(unknown_mapping)

units_weights_clean = all_weights.groupby('NormalizedValueUOMConcept')\
   .size().reset_index()

lbs_ll =  90
lbs_ul = 700

In [14]:
all_weights['NormalizedValueNumeric'] = \
   all_weights['NormalizedValueNumeric'].astype(float)

# Initialize the new column with NaNs
all_weights['UOM_assumed'] = np.nan

# Conditionally assign values to 'UOM_assumed' using .loc
# when pounds unit then pound
all_weights.loc[
    all_weights['NormalizedValueUOMConcept'].\
    isin(['pound (US and British)', 'pound (US)', 'pound (apothecary)']),
    'UOM_assumed'
] = 'pound'

# If UOM is known and not in the predefined list, use its value
all_weights.loc[
    (all_weights['NormalizedValueUOMConcept'] != 'unknown') &
    (~all_weights['NormalizedValueUOMConcept'].\
    isin(['pound (US and British)', 'pound (US)', 'pound (apothecary)'])),
    'UOM_assumed'
] = all_weights['NormalizedValueUOMConcept']

# Normalize to gram if within the specified range
all_weights.loc[
    (all_weights['NormalizedValueNumeric'] > lbs_ll * 453.6) &
    (all_weights['NormalizedValueNumeric'] <= lbs_ul * 453.6) &
    (all_weights['NormalizedValueUOMConcept'] == 'unknown'),
    'UOM_assumed'
] = 'gram'

# Normalize to ounce if within the specified range
all_weights.loc[
    (all_weights['NormalizedValueNumeric'] > lbs_ll * 16) &
    (all_weights['NormalizedValueNumeric'] <= lbs_ul * 16) &
    (all_weights['NormalizedValueUOMConcept'] == 'unknown'),
    'UOM_assumed'
] = 'ounce (avoirdupois)'

# Assign kilogram if value is less than 125
all_weights.loc[
    (all_weights['NormalizedValueNumeric'] < 125) &
    (all_weights['NormalizedValueUOMConcept'] == 'unknown'),
    'UOM_assumed'
] = 'kilogram'

# Step 5: Create 'pounds' column based on 'UOM_assumed' 
# and 'NormalizedValueNumeric'
all_weights['pounds'] = np.nan

all_weights.loc[
    all_weights['UOM_assumed'] == 'pound',
    'pounds'
] = all_weights['NormalizedValueNumeric']

all_weights.loc[
    all_weights['UOM_assumed'] == 'ounce (avoirdupois)',
    'pounds'
] = all_weights['NormalizedValueNumeric'] / 16

all_weights.loc[
    all_weights['UOM_assumed'] == 'kilogram',
    'pounds'
] = all_weights['NormalizedValueNumeric'] * 2.205

all_weights.loc[
    all_weights['UOM_assumed'] == 'gram',
    'pounds'
] = all_weights['NormalizedValueNumeric'] / 453.6

#  Mutate 'pounds' column - Set values to NaN if outside the plausible range
all_weights.loc[
    (all_weights['pounds'] < lbs_ll) | (all_weights['pounds'] > lbs_ul),
    'pounds'
] = np.nan

# Create 'kg' column by converting 'pounds' to kilograms
all_weights['kg'] = all_weights['pounds'] / 2.205

all_weights.head()

### Getting Delivery Record and determine Preterm Birth

In [15]:
# delivery condition
delivery_concode = snapshot.codeset_from_prose(url = "/definitions/delivery", variable_name= "conditionCodes")
#delivery_concode.head()
delivery_con = snapshot.load_filtered_table("Condition", delivery_concode, view_name = 'tbl_index_delivery_con')
print(delivery_con['PersonId'].nunique())
#delivery_con.head()

df = ps.sql("SELECT m.PersonId, m.RecordedDateTime, pm.* FROM tbl_index_delivery_con m JOIN ConditionCodeConceptMap pm on m.CodeConceptMapId = pm.Id").to_pandas()
delivery_con = match_code(df, delivery_concode)
print(len(delivery_con), len(delivery_con.PersonId.unique()))
#delivery_con.head()

# delivery procedure
delivery_procode = snapshot.codeset_from_prose(url = "/definitions/delivery", variable_name= "procedureCodes")
delivery_pro = snapshot.load_filtered_table("Procedure", delivery_procode, view_name = 'tbl_index_delivery_pro')
print(delivery_pro['PersonId'].nunique())
#delivery_pro.head()

df = ps.sql("SELECT m.PersonId, m.StartDateTime, pm.* FROM tbl_index_delivery_pro m JOIN ProcedureCodeConceptMap pm on m.CodeConceptMapId = pm.Id").to_pandas()
delivery_pro = match_code(df, delivery_procode)
delivery_pro.head()

In [16]:
delivery_con["source"] = "condition"
delivery_pro["source"] = "procedure"

In [18]:
# # getting preterm birth - Don't RUN
# preterm_concode = snapshot.codeset_from_prose(url = "/definitions/preterm-birth", variable_name= "codes")
# preterm_con = snapshot.load_filtered_table("Condition", preterm_concode, view_name = 'tbl_index_preterm')
# print(preterm_con['PersonId'].nunique())
# #delivery_con.head()

# df = ps.sql("SELECT m.PersonId, m.RecordedDateTime, pm.* FROM tbl_index_preterm m JOIN ConditionCodeConceptMap pm on m.CodeConceptMapId = pm.Id").to_pandas()
# preterm_con = match_code(df, preterm_concode)
# print(len(preterm_con), len(preterm_con.PersonId.unique()))
# preterm_con.head()

In [17]:
delivery_pro = delivery_pro.rename(columns={'StartDateTime': 'RecordedDateTime'})
priority = {"procedure": 1, "condition": 2}
combined_df = pd.concat([delivery_con, delivery_pro], ignore_index=True)

combined_df["priority"] = combined_df["source"].map(priority)

combined_df['RecordedDateTime'] = pd.to_datetime(combined_df['RecordedDateTime'])
# Sort by 'person_id' (ascending) and 'date' (ascending)
combined_df = (
    combined_df
    .sort_values(["PersonId", "RecordedDateTime", "priority"])
    .drop_duplicates(["PersonId", "RecordedDateTime"])
).reset_index(drop=True)

combined_df['RecordedDateTime'] = combined_df['RecordedDateTime'].dt.date
print(len(combined_df), len(combined_df.PersonId.unique()))

In [18]:
ect_surgery = combined_df.merge(ectopics, on='PersonId', how='inner')
print(len(ect_surgery.PersonId.unique()))

multi_surgery = combined_df.merge(multiple_df, on='PersonId', how='inner')
print(len(multi_surgery.PersonId.unique()))

In [19]:
combined_df.head()

In [22]:
# from datetime import datetime
# '''
# Define Preterm Birth!!
# Need to update this part later - YES, DON'T RUN
# '''
# #preterm_con.head()
# #len(preterm_con[preterm_con.isna().any(axis=1)])
# #preterm_con['RecordedDateTime'] = preterm_con['RecordedDateTime'].fillna(preterm_con['OnsetDateTime'])
# preterm_con["match_delivery"] = preterm_con['PersonId'].isin(combined_df['PersonId'])
# #preterm_con = preterm_con.drop(columns=['OnsetDateTime'])
# cutoff_date = datetime.strptime('2022-01-01', '%Y-%m-%d').date()
# preterm_con['RecordedDateTime'] = preterm_con['RecordedDateTime'].dt.date
# preterm_con = preterm_con[preterm_con['RecordedDateTime'] >= cutoff_date].reset_index(drop=True)
# print(len(preterm_con.dropna()))#11759 row out of 13934 missing rows
# preterm_con = preterm_con.dropna()
# #preterm_con.head()
# #len(preterm_con), len(preterm_con.PersonId.unique())
# preterm_con = preterm_con.sort_values(by=['PersonId', 'RecordedDateTime']).reset_index(drop=True)
# preterm_con = preterm_con.drop_duplicates(subset=['PersonId', 'RecordedDateTime'], keep='first').reset_index(drop=True)
# len(preterm_con), len(preterm_con.PersonId.unique())

In [20]:
from datetime import datetime
# drop the ectopic and multiple pregency
combined_df_cleaned = combined_df[~combined_df['PersonId'].isin(ectopics['PersonId'])]
combined_df_cleaned = combined_df[~combined_df['PersonId'].isin(multiple_df['PersonId'])]

#combined_df_cleaned = combined_df_cleaned.drop(columns=['OnsetDateTime'])
combined_df_cleaned = combined_df_cleaned.dropna()
cutoff_date = datetime.strptime('2022-01-01', '%Y-%m-%d').date()
combined_df_cleaned = combined_df_cleaned[combined_df_cleaned['RecordedDateTime'] >= cutoff_date].reset_index(drop=True)
#combined_df_cleaned = combined_df_cleaned.drop(columns=['index'])
print(len(combined_df_cleaned), len(combined_df_cleaned.PersonId.unique()))
combined_df_cleaned.head()

In [25]:
# '''
# Identify preterm birth label
# Need to update this part later - DON'T RUN
# '''

# def is_preterm_related(label):
#     label = label.lower()
#     if 'preterm' in label or 'premature' in label: #double check here
#         return True
#     return False

# df = delivery_con.copy()
# df['is_preterm'] = df['Code'].apply(is_preterm_related)
# df = df[df.is_preterm == True].copy()
# #df = df.drop(columns=["OnsetDateTime"])
# print(len(df), len(df.PersonId.unique()))

# df['RecordedDateTime'] = df['RecordedDateTime'].dt.date
# df = df[df['RecordedDateTime'] >= cutoff_date].reset_index(drop=True)
# print(len(df), len(df.PersonId.unique()))
# # df['preterm'] = df.PersonId.isin(preterm_con['PersonId'])
# # df['preterm'].value_counts()

# # merge df with preterm_con
# preterm_df = pd.concat([preterm_con, df]).copy()
# preterm_df = preterm_df.sort_values(by=['PersonId', 'RecordedDateTime']).reset_index(drop=True)

In [21]:
delivery_df = (
    combined_df_cleaned
    .sort_values(["PersonId", "RecordedDateTime"])
    .drop_duplicates(subset=["PersonId"], keep="first")
    [["PersonId", "RecordedDateTime", "Code"]]
    .reset_index(drop=True)
)
delivery_df.head()

In [22]:
delivery_df.shape, delivery_df.PersonId.nunique()

### Using Z code to get gestational age

In [23]:
# Z code
zcodecode = snapshot.codeset_from_prose(url = "/definitions/pregnancy-zcode", variable_name= "zcodes")
zcode = snapshot.load_filtered_table("Condition", zcodecode, view_name = 'tbl_index_zcodes')
print(zcode['PersonId'].nunique())
df = ps.sql("SELECT m.PersonId, m.RecordedDateTime, pm.* FROM tbl_index_zcodes m JOIN ConditionCodeConceptMap pm on m.CodeConceptMapId = pm.Id").to_pandas()
zcode = match_code(df, zcodecode)
print(len(zcode), len(zcode.PersonId.unique()))

In [24]:
print(len(delivery_df), delivery_df.PersonId.nunique())
mask_remove = zcode["Code"].str.lower().isin([
    "weeks of gestation of pregnancy not specified",
    "less than 8 weeks gestation of pregnancy"
])

zcode = zcode.loc[~mask_remove].copy()
zcode["gestational_week_zcode"] = (
    zcode["Code"]
    .str.extract(r"(\d+)", expand=False)
    .astype("float")
)

zcode = (
    zcode.sort_values(["PersonId", "RecordedDateTime", "gestational_week_zcode"])
        .groupby(["PersonId", "RecordedDateTime"], as_index=False)
        .tail(1)
)
zcode.gestational_week_zcode.describe()

In [25]:
zcode = zcode.rename(columns={"RecordedDateTime": "zcodetime"})
zcode = zcode.rename(columns={"Code": "zcode"})
deliv_zcode = delivery_df.merge(zcode, on='PersonId', how='inner')
print(len(deliv_zcode), deliv_zcode.PersonId.nunique())
deliv_zcode = deliv_zcode.sort_values(["PersonId", "zcodetime"])
deliv_zcode.isna().sum()

In [26]:
deliv_zcode["RecordedDateTime"] = pd.to_datetime(deliv_zcode["RecordedDateTime"], errors="coerce")
deliv_zcode["zcodetime"] = pd.to_datetime(deliv_zcode["zcodetime"], errors="coerce")

deliv_zcode = deliv_zcode[
    deliv_zcode["zcodetime"] <= deliv_zcode["RecordedDateTime"]
].copy()
deliv_zcode["diff_days"] = (
    deliv_zcode["RecordedDateTime"] - deliv_zcode["zcodetime"]
) / pd.Timedelta(days=1)
deliv_zcode = deliv_zcode[deliv_zcode["diff_days"] <= 300].copy()

deliv_zcode["zcodetime_date"] = deliv_zcode["zcodetime"].dt.date

deliv_zcode = (
    deliv_zcode
    .sort_values(["PersonId", "RecordedDateTime", "zcodetime_date", "gestational_week_zcode"])
    .groupby(["PersonId", "RecordedDateTime", "zcodetime_date"], as_index=False)
    .tail(1)
)

idx = deliv_zcode.groupby(["PersonId", "RecordedDateTime"])["diff_days"].idxmin()
closest_df = deliv_zcode.loc[idx].copy()
len(closest_df), closest_df.PersonId.nunique()

In [27]:
closest_df.diff_days.describe()

In [28]:
zcode_counts = (
    deliv_zcode
    .groupby(["PersonId", "RecordedDateTime"])
    .size()
    .reset_index(name="zcode_count")
)
print(zcode_counts.shape, zcode_counts.PersonId.nunique())
zcode_counts.head()

In [29]:
zcode_counts["zcode_count"].describe()

In [30]:
delivery_df = closest_df.copy()

delivery_df["estimated_LMP"] = (
    delivery_df["zcodetime"] -
    pd.to_timedelta(delivery_df["gestational_week_zcode"] * 7, unit="D")
)

delivery_df["gestational_week"] = (
    (delivery_df["RecordedDateTime"] - delivery_df["estimated_LMP"])
    / pd.Timedelta(days=7)
)

delivery_df.head()

In [31]:
delivery_df["gestational_age_days_at_delivery"] = (
    delivery_df["RecordedDateTime"] - delivery_df["estimated_LMP"]
).dt.days


In [32]:
delivery_df.gestational_week.describe()

In [33]:
delivery_df = delivery_df[delivery_df["gestational_week"] >= 24].copy()
delivery_df = delivery_df[delivery_df["gestational_week"] <= 42].copy()
print(delivery_df.gestational_week.describe())

delivery_df["preterm"] = (delivery_df["gestational_week"] < 37).astype(int)
print(delivery_df["preterm"].value_counts())

In [34]:
delivery_df["gestational_age_days_at_delivery"].describe()

In [35]:
delivery_df = delivery_df.merge(zcode_counts[['PersonId', 'zcode_count']], on='PersonId', how='inner')
zcode_count_df = delivery_df[['PersonId', 'zcode_count']].copy()
output_path_local = study.get_output_path(fs = True)
file_to_write = output_path_local + "/contrl_zcodecount.csv"
zcode_count_df.to_csv(file_to_write, index = False)

In [36]:
import matplotlib.pyplot as plt
mode_week = delivery_df["gestational_week"].round().mode()[0]
plt.figure()
delivery_df["gestational_week"].hist(bins=40)
plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")

plt.xlabel("Gestational age (weeks)")
plt.ylabel("Count")
plt.title("Distribution of gestational age at delivery")

plt.legend()
plt.show()

In [30]:
'''
Use the Spark Version below
'''

# # preterm_df['RecordedDateTime'] = pd.to_datetime(preterm_df['RecordedDateTime']).dt.date
# # delivery_df['RecordedDateTime'] = pd.to_datetime(delivery_df['RecordedDateTime']).dt.date
# from datetime import timedelta
# def match_preterm(row):
#     person_id = row['PersonId']
#     delivery_date = row['RecordedDateTime']
    
#     # find record before delivery
#     records = preterm_df[
#         (preterm_df['PersonId'] == person_id) &
#         (preterm_df['RecordedDateTime'] <= delivery_date + timedelta(days=30)) &
#         (preterm_df['RecordedDateTime'] >= delivery_date - timedelta(days=30)) # preterm record within 30 days of the delivery
#     ] # should preterm record == delivery_date?
    
#     if records.empty:
#         return pd.Series([None, None, False])
    
#     # keep the record closest to delivery
#     closest = records.loc[(delivery_date - records['RecordedDateTime']).idxmin()]
    
#     return pd.Series([closest['Code'], closest['RecordedDateTime'], True])

# delivery_df[['preterm_code', 'preterm_date', 'is_preterm']] = delivery_df.apply(match_preterm, axis=1)

In [28]:
# from pyspark.sql.functions import col, abs, min as spark_min, when
# from pyspark.sql.window import Window
# from pyspark.sql import functions as F

# # Assume preterm_df and delivery_df are PySpark DataFrames
# # Ensure RecordedDateTime columns are in timestamp or date format
# delivery_df = spark.createDataFrame(delivery_df)
# preterm_df = spark.createDataFrame(preterm_df)
# # Create a joined DataFrame for matching
# joined_df = delivery_df.alias("d").join(
#     preterm_df.alias("p"),
#     (col("d.PersonId") == col("p.PersonId")) &
#     (col("p.RecordedDateTime") >= F.date_sub(col("d.RecordedDateTime"), 30)) &
#     (col("p.RecordedDateTime") <= F.date_add(col("d.RecordedDateTime"), 30)),
#     how="left"
# )

# # Compute absolute difference in days between delivery and preterm record
# joined_df = joined_df.withColumn(
#     "date_diff",
#     abs(F.datediff(col("d.RecordedDateTime"), col("p.RecordedDateTime")))
# )

# # Define a window to find the closest preterm record per delivery
# window_spec = Window.partitionBy("d.PersonId", "d.RecordedDateTime").orderBy(col("date_diff").asc())

# # Add row number and filter to only keep the closest match
# closest_df = joined_df.withColumn("rn", F.row_number().over(window_spec)).filter(col("rn") == 1)

# # Select final columns and indicate whether a match was found
# final_df = closest_df.select(
#     col("d.*"),
#     col("p.Code").alias("preterm_code"),
#     col("p.RecordedDateTime").alias("preterm_date"),
#     when(col("p.Code").isNotNull(), True).otherwise(False).alias("is_preterm")
# )

# delivery_df = final_df.toPandas()
# delivery_df.is_preterm.value_counts()

In [37]:
full_control_df = delivery_df.copy()

### Checking Pregnancy Date and adding on Weight

In [38]:
# '''
# estimate the conception date
# Also need to update this later
# '''
# def estimate_conception_date(row):
#     delivery_date = row['RecordedDateTime']
#     # preterm birth 35 weeks, normal 39 weeks
#     weeks = 35 if row['is_preterm'] else 39
#     return delivery_date - pd.Timedelta(weeks=weeks)

# delivery_df['estimated_LMP'] = delivery_df.apply(estimate_conception_date, axis=1)
delivery_df.head()

In [39]:
ps.set_option("compute.ops_on_diff_frames", True)

In [40]:
'''
getting the weight record:
1. pre-pregancy - within 12 weeks before or after the date
2. pre-delivery - within 4 weeks before delivery date
'''

from pyspark.sql import functions as F
from pyspark.sql import Window

all_weights_df = all_weights[['PersonId', 'RecordedDateTime', 'pounds', 'kg']].copy()
all_weights_df['RecordedDateTime'] = all_weights_df['RecordedDateTime'].dt.date
all_weights_df = all_weights_df.dropna()
weights_df = all_weights_df.to_spark()
weights_df = weights_df.withColumn("RecordedDateTime", F.to_timestamp("RecordedDateTime"))
delivery_sdf = spark.createDataFrame(delivery_df).withColumn("RecordedDateTime", F.to_timestamp("RecordedDateTime"))

# find match PersonId in both weight and delivery
person_ids_with_weight = weights_df.select("PersonId").distinct()
person_ids_with_delivery = delivery_sdf.select("PersonId").distinct()

# get those person
valid_person_ids = person_ids_with_weight.join(person_ids_with_delivery, on="PersonId", how="inner")

# filteriong out both side of the data
delivery_sdf = delivery_sdf.join(valid_person_ids, on="PersonId", how="inner")
weights_df = weights_df.join(valid_person_ids, on="PersonId", how="inner")

all_weights_df.head()

In [41]:
delivery_df.head()

In [42]:
# all_weights_df_pd = all_weights_df.to_pandas()
# all_weights_df_pd.head()

In [46]:
from pyspark.sql.functions import col, abs, datediff, row_number, date_sub, date_add
from pyspark.sql.window import Window
from pyspark.sql import functions as F

#delivery_df = spark.createDataFrame(delivery_df)
def add_weight_info(delivery_df, all_weights_df):
    # Create conception ±12 week window
    delivery_df = delivery_df.withColumn("start_con", date_sub(col("estimated_LMP"), 7*12)) \
                             .withColumn("end_con", date_add(col("estimated_LMP"), 7*12))

    # Create delivery - 4 week window
    delivery_df = delivery_df.withColumn("start_del", date_sub(col("RecordedDateTime"), 28))

    # ------------------- Join 1: Weight around conception -------------------
    weights_con = all_weights_df.alias("w").join(
        delivery_df.alias("d"),
        (col("w.PersonId") == col("d.PersonId")) &
        (
            ((col("w.RecordedDateTime") >= col("d.start_con")) & (col("w.RecordedDateTime") < col("d.estimated_LMP"))) |
            ((col("w.RecordedDateTime") > col("d.estimated_LMP")) & (col("w.RecordedDateTime") <= col("d.end_con")))
        ),
        how="inner"
    ).withColumn(
        "date_diff_con", abs(datediff(col("w.RecordedDateTime"), col("d.estimated_LMP")))
    )

    window_con = Window.partitionBy("d.PersonId", "d.estimated_LMP").orderBy(col("date_diff_con"))
    weights_con = weights_con.withColumn("rn_con", row_number().over(window_con)).filter(col("rn_con") == 1)

    # ------------------- Join 2: Weight before delivery -------------------
    weights_del = all_weights_df.alias("w").join(
        delivery_df.alias("d"),
        (col("w.PersonId") == col("d.PersonId")) &
        (col("w.RecordedDateTime") >= col("d.start_del")) &
        (col("w.RecordedDateTime") <= col("d.RecordedDateTime")),
        how="inner"
    ).withColumn(
        "date_diff_del", abs(datediff(col("w.RecordedDateTime"), col("d.RecordedDateTime")))
    )

    window_del = Window.partitionBy("d.PersonId", "d.RecordedDateTime").orderBy(col("date_diff_del"))
    weights_del = weights_del.withColumn("rn_del", row_number().over(window_del)).filter(col("rn_del") == 1)

    # ------------------- Combine matched weights with delivery -------------------
    delivery_df_pd = delivery_df.alias("d") \
    .join(
        weights_con.select(
            col("d.PersonId").alias("PersonId"),
            col("d.estimated_LMP").alias("estimated_LMP"),
            col("w.kg").alias("prepreg_weight")
        ),
        on=["PersonId", "estimated_LMP"],
        how="left"
    ) \
    .join(
        weights_del.select(
            col("d.PersonId").alias("PersonId"),
            col("d.RecordedDateTime").alias("RecordedDateTime"),
            col("w.kg").alias("predelivery_weight")
        ),
        on=["PersonId", "RecordedDateTime"],
        how="left"
    )

    # Add has_weight flags
    result_df = delivery_df_pd.withColumn("has_prepreg_weight", col("prepreg_weight").isNotNull()) \
                         .withColumn("has_predelivery_weight", col("predelivery_weight").isNotNull())

    return result_df


In [48]:
'''
Use the Spark above 
'''
# def add_weight_info(delivery_df, all_weights_df):
#     # make sure it is the datetime type
#     # delivery_df['RecordedDateTime'] = pd.to_datetime(delivery_df['RecordedDateTime'])
#     # delivery_df['estimated_conception_date'] = pd.to_datetime(delivery_df['estimated_conception_date'])
#     # all_weights_df['RecordedDateTime'] = pd.to_datetime(all_weights_df['RecordedDateTime'])

#     # storage result
#     weight_around_con = [] # weight for prepreg - unit kg
#     has_weight_around_con = [] 
#     weight_before_del = [] # weight before delivery
#     has_weight_before_del = []

#     latest_weight_before_event = []  # weight before treatment
#     months_between_event_and_conception = []  

#     for row in delivery_df.itertuples():
#         person_id = row.PersonId
#         conception = row.estimated_conception_date
#         delivery = row.RecordedDateTime
#         event_date = row.event_date

#         person_weights = all_weights_df_pd[all_weights_df_pd['PersonId'] == person_id]

#         # check between estimated_conception_date +- 12 week "OR"
#         start_con = conception - timedelta(weeks=12)
#         end_con = conception + timedelta(weeks=12)
#         con_window = person_weights[
#         ((person_weights['RecordedDateTime'] >= start_con) & (person_weights['RecordedDateTime'] < conception)) |
#         ((person_weights['RecordedDateTime'] > conception) & (person_weights['RecordedDateTime'] <= end_con))]
#         if not con_window.empty: # if have value
#         # keep the one close to conception(the estimated_conception_date)
#             nearest_con = con_window.loc[(con_window['RecordedDateTime'] - conception).abs().idxmin()]
#             weight_around_con.append(nearest_con['kg'])
#             has_weight_around_con.append(True)
#         else:
#             weight_around_con.append(None)
#             has_weight_around_con.append(False)

#         # check for the predelivery weight, 4 weeks util delivery date
#         start_del = delivery - timedelta(weeks=4)
#         del_window = person_weights[
#             (person_weights['RecordedDateTime'] >= start_del) &
#             (person_weights['RecordedDateTime'] <= delivery)
#         ]
#         if not del_window.empty:
#             nearest_del = del_window.loc[(del_window['RecordedDateTime'] - delivery).abs().idxmin()]
#             weight_before_del.append(nearest_del['kg'])
#             has_weight_before_del.append(True)
#         else:
#             weight_before_del.append(None)
#             has_weight_before_del.append(False)

#         # find weight within 6 months before the event_date (pretreatment)
#         six_months_before_event = event_date - timedelta(days=183)
#         before_event = person_weights[
#             (person_weights['RecordedDateTime'] >= six_months_before_event) &
#             (person_weights['RecordedDateTime'] <= event_date)
#         ]
#         if not before_event.empty:
#             nearest_before_event = before_event.loc[(before_event['RecordedDateTime'] - event_date).abs().idxmin()]
#             latest_weight_before_event.append(nearest_before_event['kg'])
#         else:
#             latest_weight_before_event.append(None)

#         # get the interval event_date and estimated_conception_date (month)
#         delta_days = (conception - event_date).days
#         months_between_event_and_conception.append(round(delta_days / 30.44, 1)) 

#     # add those to df
#     delivery_df['has_prepreg_weight'] = has_weight_around_con
#     delivery_df['prepreg_weight'] = weight_around_con
#     delivery_df['has_predelivery_weight'] = has_weight_before_del
#     delivery_df['predelivery_weight'] = weight_before_del
#     delivery_df['pretreatment_weight'] = latest_weight_before_event
#     delivery_df['months_between_event_and_conception'] = months_between_event_and_conception

#     delivery_df = delivery_df[delivery_df['pretreatment_weight'].notna()]


#     return delivery_df

In [44]:
spark_weights_df = all_weights_df.to_spark()
from pyspark.sql.functions import col, to_date
spark_weights_df = spark_weights_df.withColumn("RecordedDateTime", to_date(col("RecordedDateTime"))) \
                               .filter(col("RecordedDateTime") >= "2021-01-01")
type(spark_weights_df)
#delivery_df_ps = delivery_df.to_spark()
delivery_df_ps = spark.createDataFrame(delivery_df)

In [52]:
delivery_df_pd = add_weight_info(delivery_df_ps, spark_weights_df)
# delivery_df_pd = delivery_df[['PersonId', 'RecordedDateTime', 'Code', 'source_type', 'preterm_code',
#        'preterm_date', 'is_preterm', 'estimated_conception_date',
#        'has_prepreg_weight', 'prepreg_weight', 'has_predelivery_weight',
#        'predelivery_weight']].copy()
#delivery_df_pd.show()

In [59]:
delivery_df_pd = delivery_df_pd.toPandas()
delivery_df_pd["has_predelivery_weight"].value_counts() #False     60372
delivery_df_pd["has_prepreg_weight"].value_counts() #False    142183

In [61]:
delivery_df_pd["has_predelivery_weight"].value_counts()

In [48]:
from pyspark.sql.functions import col, when

# Add 'has_both_weights' column
result_df = delivery_df_pd.withColumn(
    "has_both_weights",
    col("has_prepreg_weight") & col("has_predelivery_weight")
)

# Filter rows where both weights are available
result_df = result_df.filter(col("has_both_weights") == True)

# Add gestation_weight column
result_df = result_df.withColumn(
    "gestation_weight",
    col("predelivery_weight") - col("prepreg_weight")
)

# # Show the first few rows
# #result_df.show(10)

# # Count where has_prepreg_weight is False
# prepreg_false_count = result_df.filter(col("has_prepreg_weight") == False).count()

# # Count where has_predelivery_weight is False
# predeliv_false_count = result_df.filter(col("has_predelivery_weight") == False).count()

# print("Number of False in has_prepreg_weight:", prepreg_false_count)
# print("Number of False in has_predelivery_weight:", predeliv_false_count)
#print(result_df.count()) 107205
result_df = result_df.toPandas()
#result_df.head()

In [49]:


delivery_df_pd['has_both_weights'] = (
    delivery_df_pd['has_prepreg_weight'] &
    delivery_df_pd['has_predelivery_weight']
)
delivery_df_pd = delivery_df_pd[delivery_df_pd['has_both_weights'] == True].copy()
delivery_df_pd['gestation_weight'] = (
    delivery_df_pd['predelivery_weight'] - delivery_df_pd['prepreg_weight'])
    
delivery_df_pd.head()

In [48]:
print(delivery_df_pd.shape, delivery_df_pd.columns)

In [50]:
result_df.head()

In [52]:
preterm = result_df[result_df["Code"].str.contains("Premature|Preterm", case=False)]
preterm.PersonId.nunique()

In [53]:
result_df = result_df[
    ~result_df["PersonId"].isin(preterm["PersonId"])
]
print(result_df.shape)
result_df.head()

In [55]:
result_df = result_df.merge(zcode_counts[['PersonId', 'zcode_count']], on='PersonId', how='inner')
print(result_df.shape)

In [56]:
result_df.zcode_count.describe()

In [57]:
cols = [
    'PersonId', 'RecordedDateTime', 'estimated_LMP',
    'gestational_week', 'gestational_age_days_at_delivery', 'preterm', 'zcode_count',
    'start_del', 'prepreg_weight', 'predelivery_weight',
    'has_prepreg_weight', 'has_predelivery_weight', 'has_both_weights',
    'gestation_weight'
]

result_df = result_df[cols]

In [58]:
result_df.head()

In [59]:
result_df["gestational_week"].describe()

In [60]:
result_df.gestational_week.describe(), result_df.preterm.value_counts()

In [63]:
output_path_local = study.get_output_path(fs = True)
file_to_write = output_path_local + "/control_df_pd.csv"
delivery_df_pd.to_csv(file_to_write, header=True)

file_to_write = output_path_local + "/control_df.csv"
result_df.to_csv(file_to_write, header=True)

In [64]:
print(result_df.gestational_week.describe(), result_df.gestational_age_days_at_delivery.describe())
mode_week = result_df["gestational_week"].round().mode()[0]
median_week = result_df["gestational_week"].round(2).median()
plt.figure()
result_df["gestational_week"].hist(bins=40)
plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")
plt.axvline(median_week, linestyle="--", label=f"Median week: {median_week}")
plt.xlabel("Gestational age (weeks)")
plt.ylabel("Count")
plt.title("Distribution of gestational age at delivery")

plt.legend()
plt.show()

mode_days = result_df["gestational_age_days_at_delivery"].round().mode()[0]
median_days = result_df["gestational_age_days_at_delivery"].round(2).median()
plt.figure()
result_df["gestational_age_days_at_delivery"].hist(bins=40)
plt.axvline(mode_days, linestyle="--", label=f"Most common week: {mode_days}")
plt.axvline(median_days, linestyle="--", label=f"Median week: {median_days}")
plt.xlabel("Gestational age (days)")
plt.ylabel("Count")
plt.title("Distribution of gestational age at delivery")

plt.legend()
plt.show()

#### Adding height or BMI

In [65]:
bmi_df = snapshot.codeset_from_prose(url = "https://library.truveta.com/o/truveta/d/tr-body-mass-index-bmi", variable_name = "codes")
index_bmi_df = snapshot.load_filtered_table("Observation", bmi_df, view_name = 'tbl_index_bmi')
print(index_bmi_df['PersonId'].nunique())
index_bmi_df_lab = snapshot.load_filtered_table("LabResult", bmi_df, view_name = 'tbl_index_bmilab')
#index_bmi_df_lab

In [66]:
index_bmi_df = ps.concat([index_bmi_df, index_bmi_df_lab])
index_bmi_df = index_bmi_df[['NormalizedValueNumeric', 'PersonId', 'RecordedDateTime']].sort_values(by=['PersonId', 'RecordedDateTime']).reset_index(drop=True)

# Drop rows with missing BMI or RecordedDateTime
index_bmi_df = index_bmi_df.dropna(subset=['NormalizedValueNumeric', 'RecordedDateTime'])
index_bmi_df = index_bmi_df.drop_duplicates(subset=['PersonId', 'RecordedDateTime'], keep='first')
index_bmi_df = index_bmi_df.to_pandas()
# Convert date column to datetime type
index_bmi_df['RecordedDateTime'] = ps.to_datetime(index_bmi_df['RecordedDateTime'])
len(index_bmi_df.PersonId.unique())

In [67]:
output_path_local = study.get_output_path(fs = True)
file_to_read = output_path_local + "/control_df.csv"
delivery_df_pd = pd.read_csv(file_to_read)

delivery_df_pd = delivery_df_pd.rename(columns={'RecordedDateTime': 'delivery_date'})
# index_bmi_df = index_bmi_df.to_pandas()
# change bmi to numeric type
index_bmi_df['NormalizedValueNumeric'] = (
    index_bmi_df['NormalizedValueNumeric']
    .astype(float)
)

# remove nan or close to 0
index_bmi_df = index_bmi_df.dropna(subset=['NormalizedValueNumeric'])
index_bmi_df = index_bmi_df[index_bmi_df['NormalizedValueNumeric'] > 10.0]

In [68]:
merged_df = pd.merge(index_bmi_df, delivery_df_pd, on='PersonId')
# Pre-pregnancy BMI: BMI before conception

# change date to datetime type
merged_df['RecordedDateTime'] = pd.to_datetime(merged_df['RecordedDateTime'])
merged_df['estimated_LMP'] = pd.to_datetime(merged_df['estimated_LMP'])
merged_df['delivery_date'] = pd.to_datetime(merged_df['delivery_date'])

pre_bmi_df = merged_df.copy()
# find the closest time difference 
pre_bmi_df['TimeDiff'] = (pre_bmi_df['estimated_LMP'] - pre_bmi_df['RecordedDateTime']).abs()
pre_bmi_df = pre_bmi_df.sort_values(['PersonId', 'TimeDiff']).drop_duplicates('PersonId', keep='first')[
    ['PersonId', 'NormalizedValueNumeric']
].rename(columns={'NormalizedValueNumeric': 'PrePregnancyBMI'})


# Post-delivery BMI: first after delivery
post_bmi_df = merged_df.copy()
# find the closest bmi
post_bmi_df['TimeDiff'] = (post_bmi_df['RecordedDateTime'] - post_bmi_df['delivery_date']).abs()

post_bmi_df = post_bmi_df.sort_values(['PersonId', 'TimeDiff']).drop_duplicates('PersonId', keep='first')[
    ['PersonId', 'NormalizedValueNumeric']
].rename(columns={'NormalizedValueNumeric': 'nearDeliveryBMI'})

In [69]:
#delivery_df_pd = delivery_df_pd.drop(columns=['PrePregnancyBMI', 'nearDeliveryBMI'], errors='ignore')

delivery_df_pd = delivery_df_pd.merge(pre_bmi_df, on='PersonId', how='left')

# Merge post-delivery BMI
delivery_df_pd = delivery_df_pd.merge(post_bmi_df, on='PersonId', how='left')

In [70]:
delivery_df_pd.head()

In [74]:
print(delivery_df_pd.gestational_week.describe(), delivery_df_pd.gestational_age_days_at_delivery.describe(), 
delivery_df_pd.preterm.value_counts(), delivery_df_pd.zcode_count.describe())
mode_week = delivery_df_pd["gestational_week"].round().mode()[0]
median_week = delivery_df_pd["gestational_week"].round(2).median()
plt.figure()
delivery_df_pd["gestational_week"].hist(bins=40)
plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")
plt.axvline(median_week, linestyle="--", label=f"Median week: {median_week}")
plt.xlabel("Gestational age (weeks)")
plt.ylabel("Count")
plt.title("Distribution of gestational age at delivery")

plt.legend()
plt.show()

mode_days = delivery_df_pd["gestational_age_days_at_delivery"].round().mode()[0]
median_days = delivery_df_pd["gestational_age_days_at_delivery"].round(2).median()
plt.figure()
delivery_df_pd["gestational_age_days_at_delivery"].hist(bins=40)
plt.axvline(mode_days, linestyle="--", label=f"Most common days: {mode_days}")
plt.axvline(median_days, linestyle="--", label=f"Median days: {median_days}")
plt.xlabel("Gestational age (days)")
plt.ylabel("Count")
plt.title("Distribution of gestational age at delivery")

plt.legend()
plt.show()

In [73]:
delivery_df_pd["zcode_count"].hist(bins=40)
plt.show()

In [72]:
output_path_local = study.get_output_path(fs = True)
file_to_write = output_path_local + "/control_df.csv"
delivery_df_pd.to_csv(file_to_write, header=True)

### Outcome variables - Table 3

In [34]:
'''
if wanted this can be apply in the beginning, but for test purpose, just keep it simple
'''
def load_condition_data(
    snapshot,
    codeset_url=None,
    code_set=None,
    codes = 'codes',
    table_name="Condition",
    view_name="tbl_index_condition",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId",
    verbose=True
):
    """
    parameter:
        snapshot: Truveta snapshot 
        codeset_url: if use prose URL load codeset, fill out this
        code_set: if use manual input code snapshot.codeset(...) use this
        table_name: 'Condition', 'Procedure'
        view_name: sql table
        concept_map_table: mapping 'ConditionCodeConceptMap'
        concept_map_key: mapping key 'CodeConceptMapId'
        verbose: print out?
    
    return:
        matched_df: after mathcing pandas DataFrame
        unique_person_count: counting number of PersonId
    """
    # support two ways from prose URL load the code_set
    if code_set is None:
        if codeset_url is None:
            raise ValueError("Must provide either `codeset_url` or `code_set`")
        code_set = snapshot.codeset_from_prose(url=codeset_url, variable_name=codes)

    # temp table
    index_table = snapshot.load_filtered_table(table_name, code_set, view_name=view_name)

    if verbose:
        print(f"[{table_name}] Unique PersonId (after code filter):", index_table['PersonId'].nunique())

    # SQL match code
    sql_query = f"""
        SELECT m.PersonId, m.RecordedDateTime, pm.* 
        FROM {view_name} m 
        JOIN {concept_map_table} pm ON m.{concept_map_key} = pm.Id
    """
    df = ps.sql(sql_query).to_pandas()

    # getting the code
    matched_df = match_code(df, code_set)

    if verbose:
        print("Total matched rows:", len(matched_df))
        print("Unique PersonId (after match):", matched_df['PersonId'].nunique())

    return matched_df, matched_df['PersonId'].nunique()

In [35]:
# Gestational diabetes
# defGestationalDiabetes = import "https://library.truveta.com/o/truveta-research/d/gestational-diabetes" change to this one
# gest_diabet_code = snapshot.codeset_from_prose(url = "https://library.truveta.com/o/truveta-research/d/gestational-diabetes", variable_name= "codes")
# index_gest_diabet = snapshot.load_filtered_table("Condition", gest_diabet_code, view_name = 'tbl_index_gest_diabet')
# print(index_gest_diabet['PersonId'].nunique())

# df = ps.sql("SELECT m.PersonId, m.RecordedDateTime, pm.* FROM tbl_index_gest_diabet m JOIN ConditionCodeConceptMap pm on m.CodeConceptMapId = pm.Id").to_pandas()
# gest_diabet = match_code(df, gest_diabet_code)
# print(len(gest_diabet), len(gest_diabet.PersonId.unique()))

url = "https://library.truveta.com/o/truveta-research/d/gestational-diabetes"
gest_diabet, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    view_name="tbl_index_gest_diabet",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)

In [36]:
# defType2Diabetes = include "/definitions/type-2-diabetes"
url = "/definitions/type-2-diabetes"
type2_diabet, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    view_name="tbl_index_type2_diabet",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)
type2_diabet.head()

In [37]:
# Gestational hypertension 
# defTrGestationalHypertension = import "https://library.truveta.com/o/truveta-research/d/tr-gestational-hypertension"
#gest_hyper_code = snapshot.codeset_from_prose(url = "https://library.truveta.com/o/truveta-research/d/tr-gestational-hypertension", variable_name= "codes")
url = "https://library.truveta.com/o/truveta-research/d/tr-gestational-hypertension"
gest_hyper, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    view_name="tbl_index_gest_hyper",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)

In [38]:
# defHypertension = include "/definitions/hypertension"
url = "/definitions/hypertension"
hypertension, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    codes = 'conditionCodes', 
    view_name="tbl_index_hyer",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)
hypertension.head()

In [39]:
# Preeclampsia 
# defTrPreeclampsia = import "https://library.truveta.com/o/truveta-research/d/tr-preeclampsia"
#preeclampsia_code = snapshot.codeset_from_prose(url = "https://library.truveta.com/o/truveta-research/d/tr-preeclampsia", variable_name= "codes")
url =  "/definitions/preeclampsia"
preeclampsia, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    view_name="tbl_index_preeclampsia",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)

In [40]:
# C-section
# defTrCesareanDeliveryProcedures = import "https://library.truveta.com/o/truveta-research/d/tr-cesarean-delivery-procedures"

csection_code = snapshot.codeset_from_prose(url = "/definitions/c-section", variable_name = "codes")
index_c = snapshot.load_filtered_table("Procedure", csection_code, view_name = 'tbl_index_c')
print(index_c['PersonId'].nunique())
df = ps.sql("SELECT p.PersonId, p.RecordedDateTime, p.StartDateTime, pm.* FROM tbl_index_c p JOIN ProcedureCodeConceptMap pm on p.CodeConceptMapId = pm.Id").to_pandas()
csection = match_code(df, csection_code)
#procedure_full.head()
# case_names = surgerycodes_df.ConceptName.to_pandas().tolist()
# procedure_full = procedure_full[procedure_full['Code'].isin(case_names)]
#procedure_full.head()
csection['StartDateTime'] = csection['StartDateTime'].fillna(csection['RecordedDateTime'])
csection = csection.drop(columns=['RecordedDateTime'])
csection = csection.rename(columns={'StartDateTime': 'RecordedDateTime'})
csection.head()

In [41]:
# # Large-for-gestational-age infants P08.0, P08.1
# large_gest_age = snapshot.codeset('ICD10CM', 'self', 'P08.0', 'P08.1')
# large_gest, count = load_condition_data(
#     snapshot,
#     code_set=large_gest_age,
#     table_name="Condition",
#     view_name="tbl_index_large_gest_age"
# )

# # Small-for-gestational-age infants	P05.1
# small_gest_age = snapshot.codeset('ICD10CM', 'selfAndDescendants', 'P05.1') # self or selfandascendent
# small_gest, count = load_condition_data(
#     snapshot,
#     code_set=small_gest_age,
#     table_name="Condition",
#     view_name="tbl_index_small_gest_age"
# )

defExcessiveFetalWeight = "/definitions/excessive-fetal-weight"
excessive_fetal_weight, count = load_condition_data(
    snapshot, 
    codeset_url=defExcessiveFetalWeight,
    table_name="Condition",
    view_name="tbl_index_efw",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)

# Intrauterine growth restriction O36.59
intra_grow_restrict_code = snapshot.codeset('ICD10CM', 'selfAndDescendants', 'O36.59', 'Z36.4', 'O36.5990', 'O36.591', 'O36.592', 'O36.593', 'O36.599') # self or selfandascendent
intra_grow_restrict, count = load_condition_data(
    snapshot,
    code_set=intra_grow_restrict_code,
    table_name="Condition",
    view_name="tbl_index_intra_grow_restrict"
)

In [42]:
#Parity primiparous
primiparous = snapshot.codeset('ICD10CM', 'selfAndDescendants', 'Z34.00', 'O09.611', 'O09.511', 'O09.512', 'O09.513') # self or selfandascendent
primiparous, count = load_condition_data(
    snapshot,
    code_set=primiparous,
    table_name="Condition",
    view_name="tbl_index_primiparous"
)

#Parity multiparous
multiparous = snapshot.codeset('ICD10CM', 'selfAndDescendants', 'O34.21', 'O09.521', 'O09.40', 'O09.621', 'O09.41', 'O09.42', 'O09.43') # self or selfandascendent
multiparous, count = load_condition_data(
    snapshot,
    code_set=multiparous,
    table_name="Condition",
    view_name="tbl_index_multiparous"
)

In [43]:
defStillbirthCodeSet = "https://library.truveta.com/o/truveta/d/stillbirth-code-set"
stillbirth, count = load_condition_data(
    snapshot, 
    codeset_url=defStillbirthCodeSet,
    table_name="Condition",
    view_name="tbl_index_sb",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)

In [44]:
# check if those condition happens during pregnancy
def mark_condition_in_pregnancy(condition_df, delivery_df, col_name):
    """
    check if condition happen in preg, mark T/F to condition_col_name
    parameter:
        condition_df: condition record - table 3 variables
        delivery_df: PersonId, estimated_LMP, delivery_date, record for delivery/preg
        col_name
        
    return updated delivery_df with new col T/F
    """
    merged = condition_df.merge(
        delivery_df[['PersonId', 'estimated_LMP', 'delivery_date']],
        on='PersonId',
        how='left'
    )

    #check
    merged['in_pregnancy'] = (
        (merged['RecordedDateTime'] >= merged['estimated_LMP']) &
        (merged['RecordedDateTime'] <= merged['delivery_date'])
    )

    # keep satisfied PersonId
    flagged_ids = merged.loc[merged['in_pregnancy'], 'PersonId'].drop_duplicates()
    condition_flag = pd.DataFrame({ 'PersonId': flagged_ids, col_name: True })

    # merge back delivery_df others are False
    delivery_df = delivery_df.merge(condition_flag, on='PersonId', how='left')
    delivery_df[col_name] = delivery_df[col_name].fillna(False)

    return delivery_df

In [45]:
#defHyperlipidemia = import "https://library.truveta.com/o/truveta-research/d/hyperlipidemia"
url = "https://library.truveta.com/o/truveta-research/d/hyperlipidemia"
hyperlipidemia, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    view_name="tbl_index_hyerlip",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)
hyperlipidemia.head()

In [46]:
# # defObstructiveSleepApnea = import "https://library.truveta.com/o/truveta-research/d/obstructive-sleep-apnea"
# url = "https://library.truveta.com/o/truveta-research/d/obstructive-sleep-apnea"
# osa, count = load_condition_data(
#     snapshot, 
#     codeset_url=url,
#     table_name="Condition",
#     codes="osaCodes",
#     view_name="tbl_index_osa",
#     concept_map_table="ConditionCodeConceptMap",
#     concept_map_key="CodeConceptMapId"
# )
# osa.head()


In [47]:
#defMajorDepression = import "https://library.truveta.com/o/truveta-research/d/major-depression"
url = "https://library.truveta.com/o/truveta-research/d/major-depression"
depression, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    view_name="tbl_index_mdep",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)
depression.head()

In [19]:
output_path_local = study.get_output_path(fs = True)
file_to_read = output_path_local + "/control_df.csv"
delivery_df_pd = pd.read_csv(file_to_read)
delivery_df_pd.head()

In [20]:
delivery_df_t3 = mark_condition_in_pregnancy(gest_diabet, delivery_df_pd, 'gest_diabet')
delivery_df_t3 = mark_condition_in_pregnancy(gest_hyper, delivery_df_t3, 'gest_hyper')
delivery_df_t3 = mark_condition_in_pregnancy(preeclampsia, delivery_df_t3, 'preeclampsia')
delivery_df_t3 = mark_condition_in_pregnancy(csection, delivery_df_t3, 'csection')
delivery_df_t3 = mark_condition_in_pregnancy(excessive_fetal_weight, delivery_df_t3, 'excessive_fetal_weight')
# delivery_df_t3 = mark_condition_in_pregnancy(small_gest, delivery_df_t3, 'small_gest_age')
delivery_df_t3 = mark_condition_in_pregnancy(intra_grow_restrict, delivery_df_t3, 'intra_grow_restrict')

delivery_df_t3.head()

In [49]:
full_control_df = full_control_df.rename(columns={
    'RecordedDateTime': 'delivery_date'
})
full_control_df = mark_condition_in_pregnancy(gest_diabet, full_control_df, 'gest_diabet')
full_control_df = mark_condition_in_pregnancy(gest_hyper, full_control_df, 'gest_hyper')
full_control_df = mark_condition_in_pregnancy(preeclampsia, full_control_df, 'preeclampsia')
full_control_df = mark_condition_in_pregnancy(csection, full_control_df, 'csection')
full_control_df = mark_condition_in_pregnancy(excessive_fetal_weight, full_control_df, 'excessive_fetal_weight')
full_control_df = mark_condition_in_pregnancy(intra_grow_restrict, full_control_df, 'intra_grow_restrict')

In [21]:
delivery_df_t3['delivery_type'] = np.where(delivery_df_t3['csection'] == True, 'C-section', 'Vaginal')

# # baby weight
# conditions = [
#     delivery_df_t3['large_gest_age'] == True,
#     delivery_df_t3['small_gest_age'] == True
# ]
# choices = ['Large', 'Small']

# delivery_df_t3['infant_gest_age_class'] = np.select(conditions, choices, default='Average')
delivery_df_t3['infant_gest_age_class'] = np.where(delivery_df_t3['excessive_fetal_weight'] == True, 'Excessive', 'Average')
delivery_df_t3.infant_gest_age_class.value_counts(), delivery_df_t3.delivery_type.value_counts()

In [50]:
full_control_df['delivery_type'] = np.where(full_control_df['csection'] == True, 'C-section', 'Vaginal')
full_control_df['infant_gest_age_class'] = np.where(full_control_df['excessive_fetal_weight'] == True, 'Excessive', 'Average')
full_control_df['parity'] = 'Unknown'
full_control_df.loc[full_control_df['PersonId'].isin(primiparous['PersonId']), 'parity'] = 'Primiparous'
full_control_df.loc[full_control_df['PersonId'].isin(multiparous['PersonId']), 'parity'] = 'Multiparous'

In [22]:
delivery_df_t3['parity'] = 'Unknown'
delivery_df_t3.loc[delivery_df_t3['PersonId'].isin(primiparous['PersonId']), 'parity'] = 'Primiparous'
delivery_df_t3.loc[delivery_df_t3['PersonId'].isin(multiparous['PersonId']), 'parity'] = 'Multiparous'

In [51]:
def condition_before_pregnancy(condition_df, delivery_df, condition_col_name):
    # merge with conception date
    merged = condition_df.merge(
        delivery_df[['PersonId', 'estimated_LMP']],
        on='PersonId',
        how='left'
    )

    # check if condition happened before pregnancy
    merged['pre_pregnancy'] = (
        merged['RecordedDateTime'] < merged['estimated_LMP']
    )

    # find people with condition before pregnancy
    before_preg = merged[merged['pre_pregnancy']].drop_duplicates(subset='PersonId')

    # get list of IDs
    pre_preg_ids = before_preg['PersonId'].unique()

    # add flag to delivery_df
    delivery_df[condition_col_name] = delivery_df['PersonId'].isin(pre_preg_ids)

    return delivery_df


In [24]:
delivery_df_t3 = condition_before_pregnancy(type2_diabet, delivery_df_t3, 't2d_before_pregnancy')
delivery_df_t3 = condition_before_pregnancy(hypertension, delivery_df_t3, 'hyper_before_pregnancy')

delivery_df_t3 = condition_before_pregnancy(hyperlipidemia, delivery_df_t3, 'hyperlipid')
# delivery_df_t3 = condition_before_pregnancy(osa, delivery_df_t3, 'osa')
delivery_df_t3 = condition_before_pregnancy(depression, delivery_df_t3, 'depression')

delivery_df_t3.t2d_before_pregnancy.value_counts(), delivery_df_t3.hyper_before_pregnancy.value_counts()


In [52]:
full_control_df = condition_before_pregnancy(type2_diabet, full_control_df, 't2d_before_pregnancy')
full_control_df = condition_before_pregnancy(hypertension, full_control_df, 'hyper_before_pregnancy')
full_control_df = condition_before_pregnancy(hyperlipidemia, full_control_df, 'hyperlipid')
full_control_df = condition_before_pregnancy(depression, full_control_df, 'depression')
full_control_df['gest_diabetes_no_prior_t2d'] = (
    full_control_df['gest_diabet'] & ~full_control_df['t2d_before_pregnancy']
)

full_control_df['gest_hyper_no_prior_hyper'] = (
    full_control_df['gest_hyper'] & ~full_control_df['hyper_before_pregnancy']
)
# have preeclampsia but have no hyper before preg
full_control_df['preeclampsia_no_prior_hyper'] = (
    full_control_df['preeclampsia'] & ~full_control_df['hyper_before_pregnancy']
)

In [25]:
# have gestional diabetes but have no t2d before preg
delivery_df_t3['gest_diabetes_no_prior_t2d'] = (
    delivery_df_t3['gest_diabet'] & ~delivery_df_t3['t2d_before_pregnancy']
)

# have gestional hypertension but have no hyper before preg
delivery_df_t3['gest_hyper_no_prior_hyper'] = (
    delivery_df_t3['gest_hyper'] & ~delivery_df_t3['hyper_before_pregnancy']
)
# have preeclampsia but have no hyper before preg
delivery_df_t3['preeclampsia_no_prior_hyper'] = (
    delivery_df_t3['preeclampsia'] & ~delivery_df_t3['hyper_before_pregnancy']
)
delivery_df_t3.gest_diabetes_no_prior_t2d.value_counts(), delivery_df_t3.gest_hyper_no_prior_hyper.value_counts(), delivery_df_t3.preeclampsia_no_prior_hyper.value_counts()

In [26]:
delivery_df_t3.head()

In [53]:
prior_Csection = snapshot.codeset("ICD10CM",
  "selfAndDescendants",
  "O34.212",
  "O34.211",
  "O34.21",
  "O34.219") # self or selfandascendent
prior_Csection, count = load_condition_data(
    snapshot,
    code_set=prior_Csection,
    table_name="Condition",
    view_name="tbl_index_prior_Csection"
)

Prior_Preterm_Birth = snapshot.codeset("ICD10CM",
  "selfAndDescendants","Z87.51",
  "O09.21",
  "O09.211",
  "O09.212",
  "O09.213",
  "O09.219",) # self or selfandascendent
Prior_Preterm_Birth, count = load_condition_data(
    snapshot,
    code_set=Prior_Preterm_Birth,
    table_name="Condition",
    view_name="tbl_index_Prior_Preterm_Birth"
)

In [28]:
# delivery_df_t3 = condition_before_pregnancy(prior_Csection, delivery_df_t3, 'prior_Csection')
# delivery_df_t3 = condition_before_pregnancy(Prior_Preterm_Birth, delivery_df_t3, 'Prior_Preterm_Birth')
delivery_df_t3["prior_Csection"] = delivery_df_t3["PersonId"].isin(prior_Csection["PersonId"])
delivery_df_t3["Prior_Preterm_Birth"] = delivery_df_t3["PersonId"].isin(Prior_Preterm_Birth["PersonId"])
delivery_df_t3.head()

In [54]:
full_control_df["prior_Csection"] = full_control_df["PersonId"].isin(prior_Csection["PersonId"])
full_control_df["Prior_Preterm_Birth"] = full_control_df["PersonId"].isin(Prior_Preterm_Birth["PersonId"])

In [29]:
delivery_df_t3.prior_Csection.value_counts()

In [30]:
delivery_df_t3.Prior_Preterm_Birth.value_counts()

In [33]:
stillbirth.head()

In [36]:
delivery_df_t3 = delivery_df_t3[
    ~delivery_df_t3["PersonId"].isin(stillbirth["PersonId"])
]
delivery_df_t3["PersonId"].nunique()

In [55]:
full_control_df = full_control_df[
    ~full_control_df["PersonId"].isin(stillbirth["PersonId"])
]
full_control_df["PersonId"].nunique()

### Getting Person Info, DOB, age, income, race

In [56]:
df = ps.sql("SELECT * FROM Person").to_pandas()
person = decode_concepts(df)
#person.head()
df = ps.sql("SELECT * FROM PersonRace").to_pandas()
race = decode_concepts(df)
#race.head()
len(person.Id.unique()),len(person.Id)

In [38]:
person = person.rename(columns={'Id': 'PersonId'})
delivery_df_t3 = delivery_df_t3.merge(person[['PersonId','BirthDateTime','Ethnicity','Gender']], on='PersonId', how='left')
delivery_df_t3 = delivery_df_t3.merge(race[['PersonId','Race']], on='PersonId', how='left')
#delivery_df_t3.head()

In [57]:
person = person.rename(columns={'Id': 'PersonId'})
full_control_df = full_control_df.merge(person[['PersonId','BirthDateTime','Ethnicity','Gender']], on='PersonId', how='left')
full_control_df = full_control_df.merge(race[['PersonId','Race']], on='PersonId', how='left')

In [39]:
# age calucation 
def calculate_age(event_date, dob):
    return (event_date - dob).days / 365.25

# change to datetime.date
delivery_df_t3['delivery_date'] = pd.to_datetime(delivery_df_t3['delivery_date']).dt.date
delivery_df_t3['BirthDateTime'] = pd.to_datetime(delivery_df_t3['BirthDateTime']).dt.date

# assume merged have DOB、surgery_date、med_date、RecordedDateTime
delivery_df_t3['age_at_delivery'] = delivery_df_t3.apply(lambda row: calculate_age(row['delivery_date'], row['BirthDateTime']), axis=1)
#first take medication and surgery
#delivery_df_t3.head()

In [58]:

# age calucation 
def calculate_age(event_date, dob):
    return (event_date - dob).days / 365.25

# change to datetime.date
full_control_df['delivery_date'] = pd.to_datetime(full_control_df['delivery_date']).dt.date
full_control_df['BirthDateTime'] = pd.to_datetime(full_control_df['BirthDateTime']).dt.date

# assume merged have DOB、surgery_date、med_date、RecordedDateTime
full_control_df['age_at_delivery'] = full_control_df.apply(lambda row: calculate_age(row['delivery_date'], row['BirthDateTime']), axis=1)

In [59]:
def combine_race_ethnicity(row):
    race = str(row['Race']).strip().lower()
    ethnicity = str(row['Ethnicity']).strip().lower()

    if pd.isna(race) or pd.isna(ethnicity):
        return 'Unknown'

    if ethnicity == 'hispanic or latino':
        return 'Hispanic'

    elif ethnicity == 'not hispanic or latino':
        if race == 'white':
            return 'Non-Hispanic White'
        elif race == 'black or african american':
            return 'Non-Hispanic Black'
        elif race in ['asian', 'american indian or alaska native',
                      'native hawaiian or other pacific islander', 'other race']:
            return 'Other'
        else:
            return 'Unknown'

    else:
        return 'Unknown'


In [41]:
delivery_df_t3['race_ethnicity'] = delivery_df_t3.apply(combine_race_ethnicity, axis=1)

In [60]:
full_control_df['race_ethnicity'] = full_control_df.apply(combine_race_ethnicity, axis=1)

In [42]:
delivery_df_t3.race_ethnicity.value_counts()

##### adding income

In [61]:
df = ps.sql("SELECT * FROM SocialDeterminantsOfHealth").to_pandas()
soh = decode_concepts(df)
soh.head()

In [62]:
dd = snapshot.get_data_dictionary()
dd.loc[dd['table'] == 'SocialDeterminantsOfHealth']

sql = """
SELECT 
* 
FROM SocialDeterminantsOfHealth
"""
SDOH = snapshot.load_sql_table(sql, cache = True, 
   view_name = 'tbl_sdoh')

sql = """
SELECT 
* 
FROM Concept
"""

concept = snapshot.load_sql_table(sql, cache = True, 
   view_name = 'tbl_concept')


# concept.head()

sql = """
SELECT *
FROM tbl_concept
WHERE ConceptClass = 'SDOHAttribute';
"""

SDOH_concepts = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_sdoh_concepts')

# SDOH_concepts

sql = """
SELECT ConceptId, ConceptName
FROM tbl_concept;
"""

concept_sel = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_concept_sel')

# concept_sel.head()

# Spark SQL option

sql = """
SELECT
    agg.AttributeConceptId,
    agg.count,
    c.*
FROM
    (SELECT
         AttributeConceptId,
         COUNT(*) as count
     FROM
         tbl_sdoh
     GROUP BY
         AttributeConceptId) agg
LEFT JOIN
    tbl_concept_sel c
ON
    agg.AttributeConceptId = c.ConceptId
ORDER BY
    agg.count DESC
"""

attribute_counts = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_attribute_counts')

attribute_counts_sorted = attribute_counts\
 .sort_values(by='count', ascending=False)

# display(attribute_counts_sorted)

In [63]:

sql = """
SELECT
    agg.NormalizedValueConceptId,
    agg.count,
    c.*
FROM
    (SELECT
         NormalizedValueConceptId,
         COUNT(*) as count
     FROM
         tbl_sdoh
     GROUP BY
         NormalizedValueConceptId) agg
LEFT JOIN
    tbl_concept_sel c
ON
    agg.NormalizedValueConceptId = c.ConceptId
ORDER BY
    agg.count DESC
"""

normalized_value_counts = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_anormalized_value_counts')

normalized_value_counts_sorted = normalized_value_counts.\
   sort_values(by='count', ascending=False)

normalized_value_counts_sorted.head(10)

In [64]:
# Pandas on Spark option

# source_concept_id = SDOH.groupby('SourceConceptId').size().reset_index().copy()
# source_concept_id = source_concept_id.merge(concept, how = 'left', 
#                                                   left_on = 'SourceConceptId',
#                                                   right_on = 'ConceptId').sort_values(0, ascending=False)
                                                  

# source_concept_id.head(10)

#############################################################
#############################################################

# Spark SQL option

sql = """
SELECT
    agg.SourceConceptId,
    agg.count,
    c.*
FROM
    (SELECT
         SourceConceptId,
         COUNT(*) as count
     FROM
         tbl_sdoh
     GROUP BY
         SourceConceptId) agg
LEFT JOIN
    tbl_concept_sel c
ON
    agg.SourceConceptId = c.ConceptId
ORDER BY
    agg.count DESC
"""

source_concept_counts = snapshot.load_sql_table(sql, cache = True, 
   view_name = 'tbl_source_concept_counts')

source_concept_counts_sorted = source_concept_counts\
   .sort_values(by='count', ascending=False)

source_concept_counts_sorted.head()

In [65]:
sql = """
SELECT 
    PersonId, 
    EffectiveStartDateTime, 
    AttributeConceptId, 
    NormalizedValueNumeric, 
    NormalizedValueConceptId
FROM 
    tbl_sdoh;
"""

SDOH_2 = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_sdoh_2')
   
# SDOH_2.head(3)

sql = """
SELECT
    tbl_sdoh_2.PersonId,
    tbl_sdoh_2.EffectiveStartDateTime,
    tbl_sdoh_2.NormalizedValueNumeric,
    tbl_sdoh_2.NormalizedValueConceptId,
    tbl_concept_sel.ConceptId,
    tbl_concept_sel.ConceptName AS Attribute
FROM
    tbl_sdoh_2
LEFT JOIN tbl_concept_sel ON
    tbl_sdoh_2.AttributeConceptId = tbl_concept_sel.ConceptId;
"""

SDOH_3 = snapshot.load_sql_table(sql, cache = True, view_name = 'tbl_sdoh_3')
# SDOH_3.head(3)

sql = """
SELECT
    s.PersonId,
    s.EffectiveStartDateTime,
    s.Attribute,
    c.ConceptName AS Attribute_Value_Categorical,
    s.NormalizedValueNumeric AS Attribute_Value_Numeric
FROM
    tbl_sdoh_3 s
LEFT JOIN tbl_concept_sel c ON
    s.NormalizedValueConceptId = c.ConceptId;

"""

SDOH_4 = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_sdoh_4')
   
#SDOH_4.head(3)

sql = """
WITH SortedData AS (
    SELECT *,
           ROW_NUMBER() OVER(PARTITION BY PersonId, 
           Attribute ORDER BY EffectiveStartDateTime DESC, Attribute) 
             AS rn
    FROM tbl_sdoh_4
)
SELECT *
FROM SortedData
WHERE rn = 1
ORDER BY PersonId, EffectiveStartDateTime, Attribute;
"""

SDOH_5 = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_sdoh_5')
   
SDOH_5.head(10)

In [66]:
SDOH_5_p = SDOH_5.to_pandas()

SDOH_wide = SDOH_5_p.pivot(index='PersonId', columns='Attribute', 
   values=['EffectiveStartDateTime', 'Attribute_Value_Categorical', 
   'Attribute_Value_Numeric']).reset_index()


SDOH_wide.columns = SDOH_wide.columns.map(lambda index: f'{index[0]}_{index[1]}')
SDOH_wide.rename({'PersonId_': 'PersonId'}, axis=1, inplace=True)

#SDOH_wide.head(10)
person_p = person.copy()

person_SDOH = person_p.merge(SDOH_wide, 
    how = 'left', 
    left_on = 'PersonId', 
    right_on = 'PersonId')

# person_SDOH.head()

In [67]:
person_SDOH.columns

In [50]:
delivery_df_t3 = delivery_df_t3.merge(person_SDOH[['PersonId','Attribute_Value_Categorical_EstimatedAnnualIncome']], on='PersonId', how='left')
delivery_df_t3['Attribute_Value_Categorical_EstimatedAnnualIncome'] = delivery_df_t3['Attribute_Value_Categorical_EstimatedAnnualIncome'].fillna('Unknown')

In [68]:
full_control_df = full_control_df.merge(person_SDOH[['PersonId','Attribute_Value_Categorical_EstimatedAnnualIncome']], on='PersonId', how='left')
full_control_df['Attribute_Value_Categorical_EstimatedAnnualIncome'] = full_control_df['Attribute_Value_Categorical_EstimatedAnnualIncome'].fillna('Unknown')

In [1]:
## changing income interval
def reclassify_income(bracket):
    try:
        low = int(bracket.split('-')[0].replace(',', '').strip())
        if low <= 50000:
            return '≤50000'
        elif low <= 80000:
            return '50001-80000'
        else:
            return '>80000'
    except:
        return 'Unknown'

delivery_df_t3['Income'] = delivery_df_t3['Attribute_Value_Categorical_EstimatedAnnualIncome'].apply(reclassify_income)
# delivery_df_t3['pretreatment_weight'] = delivery_df_t3['pretreatment_weight'].fillna('Unknown')
delivery_df_t3.head()

In [69]:
## changing income interval
def reclassify_income(bracket):
    try:
        low = int(bracket.split('-')[0].replace(',', '').strip())
        if low <= 50000:
            return '≤50000'
        elif low <= 80000:
            return '50001-80000'
        else:
            return '>80000'
    except:
        return 'Unknown'

full_control_df['Income'] = full_control_df['Attribute_Value_Categorical_EstimatedAnnualIncome'].apply(reclassify_income)
# delivery_df_t3['pretreatment_weight'] = delivery_df_t3['pretreatment_weight'].fillna('Unknown')
full_control_df.head()

In [70]:
'''
this cell is the same as the first cell for ''Outcome Variable - Table 3''
if wanted this can be apply in the beginning, but for test purpose, just keep it simple
'''
def load_condition_data(
    snapshot,
    codeset_url=None,
    code_set=None,
    codes = 'codes',
    table_name="Condition",
    view_name="tbl_index_condition",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId",
    verbose=True
):
    """
    parameter:
        snapshot: Truveta snapshot 
        codeset_url: if use prose URL load codeset, fill out this
        code_set: if use manual input code snapshot.codeset(...) use this
        table_name: 'Condition', 'Procedure'
        view_name: sql table
        concept_map_table: mapping 'ConditionCodeConceptMap'
        concept_map_key: mapping key 'CodeConceptMapId'
        verbose: print out?
    
    return:
        matched_df: after mathcing pandas DataFrame
        unique_person_count: counting number of PersonId
    """
    # support two ways from prose URL load the code_set
    if code_set is None:
        if codeset_url is None:
            raise ValueError("Must provide either `codeset_url` or `code_set`")
        code_set = snapshot.codeset_from_prose(url=codeset_url, variable_name=codes)

    # temp table
    index_table = snapshot.load_filtered_table(table_name, code_set, view_name=view_name)

    if verbose:
        print(f"[{table_name}] Unique PersonId (after code filter):", index_table['PersonId'].nunique())

    # SQL match code
    sql_query = f"""
        SELECT m.PersonId, m.RecordedDateTime, pm.* 
        FROM {view_name} m 
        JOIN {concept_map_table} pm ON m.{concept_map_key} = pm.Id
        WHERE YEAR(m.RecordedDateTime) BETWEEN 1700 AND 2262 
    """
    # the WHERE statement is a little bit diff cause date won't work
    df = ps.sql(sql_query).to_pandas()

    # getting the code
    matched_df = match_code(df, code_set)

    if verbose:
        print("Total matched rows:", len(matched_df))
        print("Unique PersonId (after match):", matched_df['PersonId'].nunique())

    return matched_df, matched_df['PersonId'].nunique()
# need to re run this because first time the variable is not avaiable due to the df out of service
# defObstructiveSleepApnea = import "https://library.truveta.com/o/truveta-research/d/obstructive-sleep-apnea"
url = "https://library.truveta.com/o/truveta-research/d/obstructive-sleep-apnea"
osa, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    codes="osaCodes",
    view_name="tbl_index_osa",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)
osa.head()

In [71]:
defPrenatal = "/definitions/prenatal"
Prenatal, count = load_condition_data(
    snapshot, 
    codeset_url=defPrenatal,
    table_name="Condition",
    codes="Prenatal",
    view_name="tbl_index_Prenatal",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)
Prenatal.head()

In [72]:
full_control_df["Obstetriccare"] = full_control_df["PersonId"].isin(Prenatal["PersonId"])
# delivery_df_t3["Obstetriccare"].value_counts()
full_control_df = condition_before_pregnancy(osa, full_control_df, 'osa')
file_to_write = output_path_local + "/full_control_df.csv"
full_control_df.to_csv(file_to_write, index = False)

In [61]:
delivery_df_t3["Obstetriccare"] = delivery_df_t3["PersonId"].isin(Prenatal["PersonId"])
# delivery_df_t3["Obstetriccare"].value_counts()
delivery_df_t3 = condition_before_pregnancy(osa, delivery_df_t3, 'osa')
file_to_write = output_path_local + "/control_t1.csv"
delivery_df_t3.to_csv(file_to_write, index = False)

In [56]:
delivery_df_t3["PersonId"].nunique()

In [58]:
import matplotlib.pyplot as plt

In [59]:
print(delivery_df_t3.gestational_week.describe(), delivery_df_t3.gestational_age_days_at_delivery.describe(), 
delivery_df_t3.preterm.value_counts(), delivery_df_t3.zcode_count.describe())
mode_week = delivery_df_t3["gestational_week"].round().mode()[0]
median_week = delivery_df_t3["gestational_week"].round(2).median()
plt.figure()
delivery_df_t3["gestational_week"].hist(bins=40)
plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")
plt.axvline(median_week, linestyle="--", label=f"Median week: {median_week}")
plt.xlabel("Gestational age (weeks)")
plt.ylabel("Count")
plt.title("Distribution of gestational age at delivery")

plt.legend()
plt.show()

mode_days = delivery_df_t3["gestational_age_days_at_delivery"].round().mode()[0]
median_days = delivery_df_t3["gestational_age_days_at_delivery"].round(2).median()
plt.figure()
delivery_df_t3["gestational_age_days_at_delivery"].hist(bins=40)
plt.axvline(mode_days, linestyle="--", label=f"Most common days: {mode_days}")
plt.axvline(median_days, linestyle="--", label=f"Median days: {median_days}")
plt.xlabel("Gestational age (days)")
plt.ylabel("Count")
plt.title("Distribution of gestational age at delivery")

plt.legend()
plt.show()

### Table 1

In [119]:
from statsmodels.formula.api import ols
! pip install tableone
from tableone import TableOne
import statsmodels.api as sm

#### Inital run, don't run now

In [124]:
table1 = delivery_df_t3.copy()
len(table1), len(table1.PersonId.unique())

In [125]:
table1.isna().sum()

In [126]:
table1 = delivery_df_t3.dropna(subset=["PrePregnancyBMI"]).copy()
table1.columns

In [128]:
t2 = table1[['PersonId', 'is_preterm',
       'prepreg_weight', 'predelivery_weight', 
       'gestation_weight', 'PrePregnancyBMI', 'nearDeliveryBMI', 'gest_diabet',
       'gest_hyper', 'preeclampsia', 'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy', 'osa',
       'hyperlipid', 'depression', 'gest_diabetes_no_prior_t2d', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper', 'age_at_delivery', 'race_ethnicity', 'Income']]
columns_df = ['is_preterm',
       'prepreg_weight', 'predelivery_weight', 
       'gestation_weight', 'PrePregnancyBMI', 'nearDeliveryBMI', 'gest_diabet',
       'gest_hyper', 'preeclampsia', 'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy',
       'hyperlipid', 'depression', 'gest_diabetes_no_prior_t2d', 'osa', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper', 'age_at_delivery', 'race_ethnicity', 'Income']

cate_df = ['is_preterm','gest_diabet',
       'gest_hyper', 'preeclampsia', 'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy', 'osa',
       'hyperlipid', 'depression', 'gest_diabetes_no_prior_t2d', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper', 'race_ethnicity', 'Income']


table1_np = TableOne(t2, columns=columns_df, categorical=cate_df, pval=False)
table1_np

#### If want a Table 1 with p-value then read in this and run

Also getting all the source type label correctly

In [41]:
output_path_local = study.get_output_path(fs = True)
file_to_read = output_path_local + "/delivery_df_t3.csv"
test_df = pd.read_csv(file_to_read)
file_to_read = output_path_local + "/drug_source_label.csv"
drug_source_label = pd.read_csv(file_to_read) # use this as the 'med' and 'med_wo_t2d' -> use this source type
med_full_df = test_df[test_df.source_type == 'med'].copy()
surgery_df = test_df[test_df.source_type == 'surgery'].copy()
print(len(med_full_df), len(surgery_df))
med_noexppreg = med_full_df.copy()
med_noexppreg = med_noexppreg.drop('source_type', axis=1)
med_noexppreg = med_noexppreg.merge(drug_source_label, on = 'PersonId', how = 'right')
med_noexppreg.source_type.value_counts()
start_drug_preg = med_full_df[~med_full_df.PersonId.isin(med_noexppreg.PersonId)].copy()
print(len(start_drug_preg))
test_df = test_df[~test_df.PersonId.isin(start_drug_preg.PersonId)].copy()
print(len(test_df))
test_df.columns

In [68]:
med_noexppreg.source_type.value_counts()

#### Table 1 for Whole Group analysis 1 - med 816, surgery, control

In [57]:
test_df = test_df[['PersonId', 'is_preterm', 'source_type',
       'prepreg_weight', 'predelivery_weight', 'weight_loss',
       'gestation_weight', 'PrePregnancyBMI', 'preTreatmentBMI', 'preeclampsia', 'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy', 'osa',
       'hyperlipid', 'depression', 'gest_diabetes_no_prior_t2d', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper', 'age_at_delivery', 'race_ethnicity', 'Income']].copy()

In [55]:
file_to_read = output_path_local + "/control_t1.csv"
table1 = pd.read_csv(file_to_read)
table1 = table1.dropna(subset=["PrePregnancyBMI"]).copy()

In [58]:
table1 = table1[['PersonId', 'is_preterm',
       'prepreg_weight', 'predelivery_weight', 
       'gestation_weight', 'PrePregnancyBMI', 'nearDeliveryBMI', 'gest_diabet',
       'gest_hyper', 'preeclampsia', 'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy', 'osa',
       'hyperlipid', 'depression', 'gest_diabetes_no_prior_t2d', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper', 'age_at_delivery', 'race_ethnicity', 'Income']].copy()

In [59]:
table1_p = pd.concat([test_df, table1], ignore_index=True)
table1_p["source_type"] = table1_p["source_type"].fillna("control")

In [60]:
t1 = table1_p.copy()
columns_df = ['is_preterm',
       'prepreg_weight', 'predelivery_weight', 'weight_loss',
       'gestation_weight', 'PrePregnancyBMI', 'preTreatmentBMI', 
       'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy', 'osa',
       'hyperlipid', 'depression', 'gest_diabetes_no_prior_t2d', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper', 'age_at_delivery', 'race_ethnicity', 'Income']

cate_df = ['is_preterm', 'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy', 'osa',
       'hyperlipid', 'depression', 'gest_diabetes_no_prior_t2d', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper',  'race_ethnicity', 'Income']

groupby = 'source_type'

table1 = TableOne(t1, columns=columns_df, categorical=cate_df, groupby = groupby, pval=True)
# table1

In [61]:
table1

In [62]:
file_to_write = output_path_local + "/results/wholetest_t1p.html"
table1.to_html(file_to_write, index = True)

In [52]:
columns_df = ['is_preterm',
       'prepreg_weight', 'predelivery_weight', 'weight_loss',
       'gestation_weight', 'PrePregnancyBMI', 'preTreatmentBMI', 
       'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy', 'osa',
       'hyperlipid', 'depression', 'gest_diabetes_no_prior_t2d', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper', 'age_at_delivery', 'race_ethnicity', 'Income']

cate_df = ['is_preterm', 'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy', 'osa',
       'hyperlipid', 'depression', 'gest_diabetes_no_prior_t2d', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper',  'race_ethnicity', 'Income']

groupby = 'source_type'

table1_mednot2d = TableOne(med_noexppreg, columns=columns_df, categorical=cate_df, groupby = groupby, pval=True)
# table1

In [53]:
table1_mednot2d

In [54]:
file_to_write = output_path_local + "/results/mednot2d_t1p.html"
table1_mednot2d.to_html(file_to_write, index = True)

### medication group Table 1

In [63]:
file_to_read = output_path_local + "/drugexporsure.csv"
drugexp_df = pd.read_csv(file_to_read)
print(len(drugexp_df))
med_full_df = med_full_df.merge(drugexp_df[['PersonId', 'ExposedInPregnancy']], on = 'PersonId')

In [65]:
columns_df = ['is_preterm',
       'prepreg_weight', 'predelivery_weight', 'weight_loss',
       'gestation_weight', 'PrePregnancyBMI', 'preTreatmentBMI', 
       'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy', 'osa',
       'hyperlipid', 'depression', 'gest_diabetes_no_prior_t2d', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper', 'age_at_delivery', 'race_ethnicity', 'Income']

cate_df = ['is_preterm', 'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy', 'osa',
       'hyperlipid', 'depression', 'gest_diabetes_no_prior_t2d', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper',  'race_ethnicity', 'Income']

groupby = 'ExposedInPregnancy'

table1_drugexp = TableOne(med_full_df, columns=columns_df, categorical=cate_df, groupby = groupby, pval=True)
# table1

In [66]:
table1_drugexp

In [67]:
file_to_write = output_path_local + "/results/medexpo_t1p.html"
table1_drugexp.to_html(file_to_write, index = True)

# Comparison Tests, run this!

### Read in all the dataset that was storage in the File Explorer

In [6]:
output_path_local = study.get_output_path(fs = True)
file_to_read = output_path_local + "/delivery_df_t3.csv"
test_df = pd.read_csv(file_to_read)
test_df.head()
# check source type if missing the med_not2d then add it 
# test_df.loc[(test_df['t2d_before_pregnancy'] == False)&(test_df['source_type'] == 'med'), 'source_type'] = 'med_wo_t2d'

In [ ]:
# ONLY run this if didn't run in Table 1 section
file_to_read = output_path_local + "/drug_source_label.csv"
drug_source_label = pd.read_csv(file_to_read) # use this as the 'med' and 'med_wo_t2d' -> use this source type
med_full_df = test_df[test_df.source_type == 'med'].copy()
surgery_df = test_df[test_df.source_type == 'surgery'].copy()
print(len(med_full_df), len(surgery_df))
med_noexppreg = med_full_df.copy()
med_noexppreg = med_noexppreg.drop('source_type', axis=1)
med_noexppreg = med_noexppreg.merge(drug_source_label, on = 'PersonId', how = 'right')
med_noexppreg.source_type.value_counts()
start_drug_preg = med_full_df[~med_full_df.PersonId.isin(med_noexppreg.PersonId)].copy()
print(len(start_drug_preg))
test_df = test_df[~test_df.PersonId.isin(start_drug_preg.PersonId)].copy()
print(len(test_df))

In [69]:
file_to_read = output_path_local + "/control_t1.csv"
control_df = pd.read_csv(file_to_read)
control_df.head()

In [70]:
print(control_df.shape)
control_df = control_df.dropna(subset=["PrePregnancyBMI"]).copy()
print(control_df.shape)

In [71]:
test_df = test_df[['PersonId', 'source_type', 'is_preterm', 'prepreg_weight',
        'predelivery_weight',
       'gestation_weight', 'PrePregnancyBMI', 'gest_diabet',
       'gest_hyper', 'preeclampsia', 'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy',
       'hyperlipid', 'osa', 'depression', 'gest_diabetes_no_prior_t2d', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper', 'age_at_delivery', 'race_ethnicity', 'Income']].copy()
       
control_df['source_type'] = 'control'
control_df = control_df[['PersonId', 'source_type', 'is_preterm', 'prepreg_weight',
        'predelivery_weight',
       'gestation_weight', 'PrePregnancyBMI', 'gest_diabet',
       'gest_hyper', 'preeclampsia', 'csection', 'excessive_fetal_weight',
       'intra_grow_restrict', 'delivery_type', 'infant_gest_age_class',
       'parity', 't2d_before_pregnancy', 'hyper_before_pregnancy',
       'hyperlipid', 'osa', 'depression', 'gest_diabetes_no_prior_t2d', 'preeclampsia_no_prior_hyper',
       'gest_hyper_no_prior_hyper', 'age_at_delivery', 'race_ethnicity', 'Income']].copy()

#### combine dataset

In [86]:
med_not2d = med_noexppreg[med_noexppreg.source_type == 'med_wo_t2d'].copy()

In [73]:
combined_df = pd.concat([test_df, control_df], ignore_index=True)
# combined_df = combined_df.fillna('Unknown')
combined_df = combined_df.rename(columns={"source_type": "group"})

In [75]:
combined_df.isna().sum()

In [74]:
combined_df.race_ethnicity.value_counts()

In [76]:
# get the binary outcomes yes/no
binary_outcomes = [
    'is_preterm', 'preeclampsia_no_prior_hyper', 'csection',
    'excessive_fetal_weight', 'intra_grow_restrict', 'gest_diabetes_no_prior_t2d',
    'gest_hyper_no_prior_hyper'
]
for col in binary_outcomes:
    if col in combined_df.columns:
        combined_df[col] = combined_df[col].astype(int)

# continuous outcomes
continuous_outcomes = [
    'prepreg_weight', 'predelivery_weight', 'gestation_weight'
]

covariates = [
    "C(group)",                # treatment group: control vs drug/surgery
    "age_at_delivery",         # continuous
    "C(race_ethnicity)",       # categorical
    "C(Income)",               # categorical
    "PrePregnancyBMI",         # continuous
    "C(parity)",               # categorical 
    "t2d_before_pregnancy",    # binary
    "hyper_before_pregnancy",  # binary
    "depression"               # binary
]

In [77]:
for col, ref_order in {
    "group": ["control", "med", "med_wo_t2d", "surgery"],
    "race_ethnicity": ["Non-Hispanic White", "Non-Hispanic Black", "Hispanic", "Other", "Unknown"],
    "Income": ["≤50000", "50001-80000", ">80000", "Unknown"],
    "parity": ["Primiparous", "Multiparous", "Unknown"]
}.items():
    combined_df[col] = pd.Categorical(combined_df[col], categories=ref_order, ordered=True)


In [78]:
def get_covariates_for_outcome(outcome, df_subset):
    covs = [
        "C(group)",
        "age_at_delivery",
        "C(race_ethnicity)",
        "C(Income)",
        "PrePregnancyBMI",
        "C(parity)",
        "t2d_before_pregnancy",
        "hyper_before_pregnancy",
        "depression"
    ]

    # remove based on context
    if outcome == "gest_diabetes_no_prior_t2d":
        covs = [c for c in covs if "t2d_before_pregnancy" not in c]
    if outcome in ["gest_hyper_no_prior_hyper", "preeclampsia_no_prior_hyper"]:
        covs = [c for c in covs if "hyper_before_pregnancy" not in c]

    # also drop any constant variables
    valid_covariates = []
    for cov in covs:
        colname = cov.split("(")[-1].split(")")[0] if "C(" in cov else cov
        if colname in df_subset.columns and df_subset[colname].nunique() > 1:
            valid_covariates.append(cov)

    return valid_covariates


In [79]:
import statsmodels.formula.api as smf

def run_models(df, outcomes, model_type="logit", output_path="model_results.html"):
    from pathlib import Path
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("<html><body><h1>Regression Model Results</h1>")

    for outcome in outcomes:
        # Drop NA rows early
        outcome_col = outcome
        df_subset = df.copy()

        # Get valid covariates for this outcome and subset
        covariates = get_covariates_for_outcome(outcome, df_subset)

        cols_needed = [outcome_col] + [c.split("(")[-1].split(")")[0] if "C(" in c else c for c in covariates]
        df_subset = df_subset.dropna(subset=cols_needed)

        if df_subset[outcome_col].nunique() < 2:
            print(f"[Skipped] Outcome {outcome} has no variation.")
            continue

        formula = f"{outcome} ~ " + " + ".join(covariates)

        try:
            if model_type == "logit":
                model = smf.logit(formula=formula, data=df_subset).fit(disp=False)
            elif model_type == "ols":
                model = smf.ols(formula=formula, data=df_subset).fit()
            else:
                raise ValueError("Unsupported model_type")

            print(f"\n===== {outcome.upper()} ({model_type}) =====")
            print(model.summary())

            extract_and_save_model_results(
                model,
                outcome_name=outcome,
                model_type=model_type,
                output_path=output_path
            )

        except Exception as e:
            print(f"[Error] Could not fit model for {outcome}: {e}")

    with open(output_path, "a", encoding="utf-8") as f:
        f.write("</body></html>")


In [82]:
def extract_and_save_model_results(model, outcome_name, model_type="logit", output_path="model_results.html"):
    import pandas as pd
    import numpy as np

    summary_df = pd.DataFrame({
        "variable": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "z_or_t": model.tvalues,
        "p_value": model.pvalues.values,
        "ci_lower": model.conf_int()[0],
        "ci_upper": model.conf_int()[1]
    })

    if model_type == "logit":
        # TAKE LOG for OR and CI
        summary_df["odds_ratio"] = np.exp(summary_df["coef"])
        summary_df["or_ci_lower"] = np.exp(summary_df["ci_lower"])
        summary_df["or_ci_upper"] = np.exp(summary_df["ci_upper"])
        # getting format
        summary_df["formatted"] = summary_df.apply(
            lambda row: f"OR={row['odds_ratio']:.2f}, ({row['or_ci_lower']:.2f}, {row['or_ci_upper']:.2f}), p={row['p_value']:.3f}",
            axis=1
        )
    else:
        # OLS β
        summary_df["formatted"] = summary_df.apply(
            lambda row: f"β={row['coef']:.2f}, ({row['ci_lower']:.2f}, {row['ci_upper']:.2f}), p={row['p_value']:.3f}",
            axis=1
        )

    summary_df["outcome"] = outcome_name
    summary_df["model_type"] = model_type

    # display not matter if want can be changed
    display_df = summary_df[["variable", "formatted", "outcome", "model_type"]]

    html_table = display_df.to_html(index=False, escape=False)

    with open(output_path, "a", encoding="utf-8") as f:
        f.write(f"<h2>Outcome: {outcome_name} ({model_type})</h2>\n")
        f.write(html_table)
        f.write("<br><hr><br>")

    return summary_df  # return results


In [55]:
# # med vs surgery - if need to run uncomment this!
# surgery_med = combined_df[combined_df["group"].isin(["med", "surgery"])].copy()
# surgery_med["group"] = pd.Categorical(surgery_med["group"], categories=["med", "surgery"])
# run_models(surgery_med, binary_outcomes, model_type="logit", output_path=output_path_local + "/med_vs_surgery_logit.html")
# run_models(surgery_med, continuous_outcomes, model_type="ols", output_path=output_path_local + "/med_vs_surgery_ols.html")

In [56]:
# control vs surgery
surgery_pair = combined_df[combined_df["group"].isin(["control", "surgery"])].copy()
surgery_pair["group"] = pd.Categorical(surgery_pair["group"], categories=["control", "surgery"])
run_models(surgery_pair, binary_outcomes, model_type="logit", output_path=  output_path_local + "/control_vs_surgery_logit.html")
run_models(surgery_pair, continuous_outcomes, model_type="ols", output_path=  output_path_local + "/control_vs_surgery_ols.html")


In [83]:
# control vs drug
med_pair = combined_df[combined_df["group"].isin(["control", "med"])].copy()
med_pair["group"] = pd.Categorical(med_pair["group"], categories=["control", "med"])
run_models(med_pair, binary_outcomes, model_type="logit", output_path=  output_path_local + "/control_vs_med_logit.html")
run_models(med_pair, continuous_outcomes, model_type="ols", output_path=  output_path_local + "/control_vs_med_ols.html")


In [84]:
med_pair.group.value_counts()

##### Run Sub analysis

In [87]:
sub_df = pd.concat([med_not2d, control_df], ignore_index=True)
# combined_df = combined_df.fillna('Unknown')
sub_df = sub_df.rename(columns={"source_type": "group"})

In [90]:
for col in binary_outcomes:
    if col in sub_df.columns:
        sub_df[col] = sub_df[col].astype(int)

In [88]:
sub_df.group.value_counts()

In [91]:
sub_df["group"] = pd.Categorical(sub_df["group"], categories=["control", "med_wo_t2d"])
run_models(sub_df, binary_outcomes, model_type="logit", output_path=  output_path_local + "/control_vs_mednot2d_logit.html")
run_models(sub_df, continuous_outcomes, model_type="ols", output_path=  output_path_local + "/control_vs_mednot2d_ols.html")

### Removing Pre-existing T2D and taking Medication

In [51]:
# naming is wrong but please just keep it
nopret2d_med_df = test_df[(test_df.t2d_before_pregnancy == False)]
control_not2d = control_df[(control_df.t2d_before_pregnancy == False)]

combinednot2d_df = pd.concat([nopret2d_med_df, control_not2d], ignore_index=True)
combinednot2d_df = combinednot2d_df.rename(columns={"source_type": "group"})

In [52]:
# update covariates, remove t2d_before_pregnancy
covariates = [
    "age_at_delivery",
    "C(race_ethnicity)",
    "C(Income)",
    "PrePregnancyBMI",
    "C(parity)",
    "hyper_before_pregnancy",
    "depression"
]

for col in binary_outcomes:
    if col in combinednot2d_df.columns:
        combinednot2d_df[col] = combinednot2d_df[col].astype(int)


In [53]:
combinednot2d_df.group.value_counts()

In [54]:
# save the no pre-exist t2d
file_to_write = output_path_local + "/nopret2d_med_df.csv"
nopret2d_med_df.to_csv(file_to_write, index = False)

In [55]:
# control vs surgery
surgerynot2d_pair = combined_df[combined_df["group"].isin(["control", "surgery"])].copy()
surgerynot2d_pair["group"] = pd.Categorical(surgerynot2d_pair["group"], categories=["control", "surgery"])
run_models(surgerynot2d_pair, binary_outcomes, model_type="logit", output_path=  output_path_local + "/control_vs_surgeryno2td_logit.html")
run_models(surgerynot2d_pair, continuous_outcomes, model_type="ols", output_path=  output_path_local + "/control_vs_surgeryno2td_ols.html")

# control vs surgery
surgery_med_t2d = combined_df[combined_df["group"].isin(["med", "surgery"])].copy()
surgery_med_t2d["group"] = pd.Categorical(surgery_med_t2d["group"], categories=["med", "surgery"])
run_models(surgery_med_t2d, binary_outcomes, model_type="logit", output_path=  output_path_local + "/med_vs_surgeryno2td_logit.html")
run_models(surgery_med_t2d, continuous_outcomes, model_type="ols", output_path=  output_path_local + "/med_vs_surgeryno2td_ols.html")

In [ ]:
mednot2d_pair = combinednot2d_df[combinednot2d_df["group"].isin(["control", "med"])].copy()
mednot2d_pair["group"] = pd.Categorical(mednot2d_pair["group"], categories=["control", "med"])
run_models(mednot2d_pair, binary_outcomes, model_type="logit", output_path=  output_path_local + "/control_vs_mednot2d_logit.html")
run_models(mednot2d_pair, continuous_outcomes, model_type="ols", output_path=  output_path_local + "/control_vs_mednot2d_ols.html")